# Task 3 — Dual Local Models Blind A/B (Google Colab / Jupyter / Kaggle Offline)

Этот блокнот — **специальная версия Task 3** для честного сравнения двух локальных моделей на одной и той же входной выборке.

Он делает следующее:

- принимает вход из **query**, **списка DOI / URL / identifier**, **commands** или **Task 1 YAML**;
- запускает **один и тот же Task 3 pipeline два раза**:
  - для **локальной модели из коробки**,
  - для **локальной дообученной модели**;
- старается использовать **одни и те же processed papers**, чтобы сравнение было максимально честным;
- собирает **blind offline A/B form**, где эксперт видит только **Variant A / Variant B** и **Hidden model α / Hidden model β**;
- сохраняет:
  - `task3_dual_local_model_review_offline_ab.html`
  - `expert_dual_model_blind_review_bundle.zip`
  - `task3_dual_local_model_blind_key.json` (**owner-only**, не передавать эксперту)

> Это обычный Jupyter notebook, который можно открывать как в Colab, так и в совместимых Jupyter-средах.


> Эта версия дополнительно адаптирована под **Kaggle Save Version** в **автономном режиме**: конфиг можно задать кодом, репозиторий ищется в `\/kaggle\/input`, а выходные артефакты по умолчанию пишутся в `\/kaggle\/working`.


# Пошаговый туториал

## Базовый сценарий
1. Сначала запустите ячейку **«Быстрый сетап до установки репозитория — графическое меню»**:
   - укажите `Task 1 YAML path` или загрузите YAML,
   - либо заполните `query + identifiers`,
   - и сразу задайте **Model α / Model β**, backend и пути к моделям.
2. Затем запустите отдельную ячейку **«Проверка быстрого сетапа до установки репозитория»**:
   - она проверит обязательные поля,
   - подскажет, что именно нужно исправить,
   - и сохранит значения в глобальный конфиг блокнота.
3. Запустите ячейку **«Установка и импорт»**.
4. Запустите ячейку **«Форма dual-local blind A/B»**:
   - быстрый сетап автоматически подтянется в форму;
   - при необходимости можно уточнить path-поля или использовать upload.
5. Нажмите **«Проверить конфиг»** в форме или сразу переходите к запуску, если всё уже валидно.
6. Запустите нижнюю ячейку **«Запуск dual-local blind A/B»**.
7. На выходе получите:
   - два отдельных Task 3 bundle,
   - blind offline HTML для эксперта,
   - owner-only key file,
   - expert ZIP без раскрытия identity key.

## Что изменилось в этой версии
- **Быстрый сетап теперь графический и вынесен в начало блокнота**: эксперт может внести YAML / identifiers / модели до установки репозитория.
- **Проверка быстрого сетапа вынесена в отдельную ячейку** и не смешана с вводом данных.
- Ошибки по-прежнему **не роняют run-cell**, а показываются в понятном блоке со списком того, что проверить.
- В долгих этапах сохранены **прогресс-бары**:
  - общий прогресс dual-run,
  - прогресс текущего pipeline,
  - детализация по paper/page/candidate там, где это доступно из runtime.

## Почему этот workflow честнее обычного
- Сначала выполняется **run α**.
- Затем **run β** использует те же `processed papers`, чтобы не скачивать статьи заново и не менять входные данные.
- Blind review не показывает эксперту, какая именно модель стоит за α / β.

## Формат commands
Можно вставить текст или загрузить `.txt / .yaml / .json`, например:

```text
query: temporal knowledge graph multimodal hypothesis generation
identifier: 10.1038/s41586-020-2649-2
identifier: https://doi.org/10.1126/science.abc1234
domain_id: science
model_a_vlm_model_id: /models/base
model_b_vlm_model_id: /models/tuned
hf_offline: true
```


## Kaggle Save Version / автономный запуск

1. Прикрепите к notebook входной датасет с архивом репозитория или уже распакованным репозиторием.
2. В следующей ячейке задайте `TASK3_DUAL_SETUP` — это основной способ для **headless**-прогона без ручного клика по widget-формам.
3. Убедитесь, что входные пути ведут в `\/kaggle\/input/...`, а `out_dir` — в `\/kaggle\/working/...`.
4. После `Save Version` Kaggle выполнит notebook сверху вниз в отдельной сессии и сохранит выходные файлы версии; именно поэтому notebook ниже подхватывает конфиг из кода и пишет manifest рядом с артефактами.


In [ ]:
# @title
# Конфиг для Kaggle Save Version / автономного прогона
# Отредактируйте этот dict, если хотите запускать notebook на Kaggle без ручной работы с widget-формами.

from pathlib import Path
import os
import json

TASK3_IS_KAGGLE_RUNTIME = Path('/kaggle/working').exists() or bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
TASK3_KAGGLE_INPUT_ROOT = Path('/kaggle/input') if Path('/kaggle/input').exists() else None
TASK3_KAGGLE_WORKING_ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path.cwd()

if TASK3_IS_KAGGLE_RUNTIME:
    os.environ.setdefault('TASK3_NOTEBOOK_PLATFORM', 'kaggle')
    os.environ.setdefault('TASK3_NOTEBOOK_KAGGLE_AUTORUN', '1')
    os.environ.setdefault('TASK3_NOTEBOOK_SKIP_OPTIONAL_DEPS', '1')
    os.environ.setdefault('TASK3_ALLOW_GIT_CLONE', '0')
    os.environ.setdefault('HF_HUB_OFFLINE', '1')
    os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
    os.environ.setdefault('HF_DATASETS_OFFLINE', '1')

# Заполняйте значения ниже под свой notebook/dataset.
# Главное правило для Kaggle:
# - входные файлы -> /kaggle/input/...
# - выходные файлы -> /kaggle/working/...
if 'TASK3_DUAL_SETUP' not in globals():
    TASK3_DUAL_SETUP = {
        'input_mode': 'query',
        'task1_yaml_path': '',
        'processed_path': '',
        'commands_file_path': '',
        'query': '',
        'identifiers': [],
        'commands_text': '',
        'expert': {
            'last_name': '',
            'first_name': '',
            'patronymic': '-',
        },
        'domain_id': 'science',
        'out_dir': str((TASK3_KAGGLE_WORKING_ROOT / 'task3_dual_local_blind_ab').resolve()) if TASK3_IS_KAGGLE_RUNTIME else 'runs/task3_dual_local_blind_ab',
        'search_limit': 25,
        'top_papers': 12,
        'top_hypotheses': 8,
        'candidate_top_k': 16,
        'top_pairs': 8,
        'annoy_n_trees': 32,
        'annoy_top_k': 6,
        'include_multimodal': True,
        'run_vlm': True,
        'edge_mode': 'auto',
        'link_prediction_backend': 'auto',
        'hf_offline': bool(TASK3_IS_KAGGLE_RUNTIME),
        'create_offline_form': True,
        'create_expert_bundle': True,
        'auto_download_offline': False,
        'auto_download_bundle': False,
        'auto_download_owner_key': False,
        'model_a': {
            'owner_label': 'base_local_model',
            'vlm_backend': 'qwen2_vl',
            'vlm_model_id': '',
            'local_text_model': '',
        },
        'model_b': {
            'owner_label': 'finetuned_local_model',
            'vlm_backend': 'qwen2_vl',
            'vlm_model_id': '',
            'local_text_model': '',
        },
    }

TASK3_DUAL_SETUP_YAML = globals().get('TASK3_DUAL_SETUP_YAML') or ''

preview = {
    'platform': 'kaggle' if TASK3_IS_KAGGLE_RUNTIME else 'local_or_colab',
    'out_dir': TASK3_DUAL_SETUP.get('out_dir'),
    'input_mode': TASK3_DUAL_SETUP.get('input_mode'),
    'processed_path': TASK3_DUAL_SETUP.get('processed_path'),
    'query': TASK3_DUAL_SETUP.get('query'),
    'hf_offline': TASK3_DUAL_SETUP.get('hf_offline'),
    'model_a_backend': (TASK3_DUAL_SETUP.get('model_a') or {}).get('vlm_backend'),
    'model_b_backend': (TASK3_DUAL_SETUP.get('model_b') or {}).get('vlm_backend'),
}
print(json.dumps(preview, ensure_ascii=False, indent=2))


In [1]:
# @title
# Быстрый сетап до установки репозитория — графическое меню
# Заполните поля и нажмите «Сохранить быстрый сетап».
# Затем отдельно запустите следующую ячейку с проверкой.

from pathlib import Path
import json
import textwrap
import hashlib
import os

try:
    import yaml
except Exception as e:
    raise RuntimeError(f'PyYAML недоступен в окружении: {type(e).__name__}: {e}')

try:
    import ipywidgets as W
    from IPython.display import display
except Exception as e:
    raise RuntimeError(f'ipywidgets недоступен в окружении: {type(e).__name__}: {e}')

def _task3_runtime_is_kaggle():
    return Path('/kaggle/working').exists() or bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or os.environ.get('TASK3_NOTEBOOK_PLATFORM') == 'kaggle'


def _task3_default_out_dir_value():
    if _task3_runtime_is_kaggle():
        return str((Path('/kaggle/working') / 'task3_dual_local_blind_ab').resolve())
    return 'runs/task3_dual_local_blind_ab'


def _task3_quick_default_setup():
    return {
        "input_mode": "query",
        "task1_yaml_path": "",
        "processed_path": "",
        "commands_file_path": "",
        "query": "",
        "identifiers": [],
        "commands_text": "",
        "expert": {
            "last_name": "",
            "first_name": "",
            "patronymic": "-",
        },
        "domain_id": "science",
        "out_dir": _task3_default_out_dir_value(),
        "search_limit": 25,
        "top_papers": 12,
        "top_hypotheses": 8,
        "candidate_top_k": 16,
        "top_pairs": 8,
        "annoy_n_trees": 32,
        "annoy_top_k": 6,
        "include_multimodal": True,
        "run_vlm": True,
        "edge_mode": "auto",
        "link_prediction_backend": "auto",
        "hf_offline": _task3_runtime_is_kaggle(),
        "create_offline_form": True,
        "create_expert_bundle": True,
        "auto_download_offline": False,
        "auto_download_bundle": False,
        "auto_download_owner_key": False,
        "model_a": {
            "owner_label": "base_local_model",
            "vlm_backend": "qwen2_vl",
            "vlm_model_id": "",
            "local_text_model": "",
        },
        "model_b": {
            "owner_label": "finetuned_local_model",
            "vlm_backend": "qwen2_vl",
            "vlm_model_id": "",
            "local_text_model": "",
        },
    }


def _task3_quick_deep_update(base, patch):
    if not isinstance(base, dict) or not isinstance(patch, dict):
        return patch
    merged = dict(base)
    for key, value in patch.items():
        if isinstance(value, dict) and isinstance(merged.get(key), dict):
            merged[key] = _task3_quick_deep_update(merged[key], value)
        else:
            merged[key] = value
    return merged


def _task3_quick_parse_yaml(text):
    text = str(text or '').strip()
    if not text:
        return {}, None
    try:
        payload = yaml.safe_load(text) or {}
        if not isinstance(payload, dict):
            raise ValueError('YAML должен содержать объект верхнего уровня')
        return payload, None
    except Exception as e:
        return {}, f'{type(e).__name__}: {e}'


def _task3_quick_normalize_uploaded_items(upload_value):
    if not upload_value:
        return []
    if isinstance(upload_value, dict):
        items = []
        for key, value in upload_value.items():
            if isinstance(value, dict):
                item = dict(value)
                item.setdefault('name', key)
                items.append(item)
        return items
    if isinstance(upload_value, (list, tuple)):
        out = []
        for item in upload_value:
            if hasattr(item, 'keys'):
                out.append(dict(item))
            elif isinstance(item, dict):
                out.append(dict(item))
        return out
    return []


def _task3_quick_save_upload(upload_value, fallback_name):
    items = _task3_quick_normalize_uploaded_items(upload_value)
    if not items:
        return ''
    item = items[0]
    raw_name = str(item.get('name') or fallback_name or 'upload.bin').strip() or fallback_name or 'upload.bin'
    safe_name = ''.join(ch if ch.isalnum() or ch in '._-' else '_' for ch in raw_name)
    target_dir = Path('/tmp/task3_quick_setup_uploads')
    target_dir.mkdir(parents=True, exist_ok=True)
    target = target_dir / safe_name
    content = item.get('content')
    if content is None and 'data' in item:
        content = item.get('data')
    if hasattr(content, 'tobytes'):
        content = content.tobytes()
    if isinstance(content, memoryview):
        content = content.tobytes()
    if isinstance(content, str):
        content = content.encode('utf-8')
    if content is None:
        raise ValueError(f'Не удалось прочитать содержимое upload: {safe_name}')
    target.write_bytes(content)
    return str(target)


def _task3_quick_render_box(title, lines=None, tone='neutral', details=''):
    palette = {
        'neutral': ('#eef2ff', '#c7d2fe', '#312e81'),
        'running': ('#eff6ff', '#bfdbfe', '#1d4ed8'),
        'success': ('#ecfdf5', '#86efac', '#166534'),
        'warning': ('#fff7ed', '#fdba74', '#9a3412'),
        'danger': ('#fef2f2', '#fca5a5', '#991b1b'),
    }
    bg, border, color = palette.get(tone, palette['neutral'])
    items = ''.join(f'<li>{str(line)}</li>' for line in (lines or []))
    body = f'<ul style="margin:8px 0 0 18px;">{items}</ul>' if items else ''
    details_html = f'<div style="margin-top:8px;"><pre style="white-space:pre-wrap; margin:0;">{details}</pre></div>' if details else ''
    return (
        f'<div style="padding:12px 14px; border-radius:12px; border:1px solid {border}; '
        f'background:{bg}; color:{color}; margin:6px 0;">'
        f'<div style="font-weight:700;">{title}</div>{body}{details_html}</div>'
    )


def _task3_quick_build_initial_setup():
    merged = _task3_quick_default_setup()
    raw_setup = globals().get('TASK3_DUAL_SETUP')
    if isinstance(raw_setup, dict):
        merged = _task3_quick_deep_update(merged, raw_setup)
    json_text = str(globals().get('TASK3_DUAL_SETUP_JSON') or os.environ.get('TASK3_DUAL_SETUP_JSON') or '').strip()
    if json_text:
        try:
            payload = json.loads(json_text)
            if isinstance(payload, dict):
                merged = _task3_quick_deep_update(merged, payload)
        except Exception:
            pass

    setup_path = str(globals().get('TASK3_DUAL_SETUP_PATH') or os.environ.get('TASK3_DUAL_SETUP_PATH') or '').strip()
    if setup_path:
        setup_file = Path(setup_path).expanduser()
        if setup_file.exists():
            try:
                payload = yaml.safe_load(setup_file.read_text(encoding='utf-8')) or {}
                if isinstance(payload, dict):
                    merged = _task3_quick_deep_update(merged, payload)
            except Exception:
                pass

    yaml_text = str(globals().get('TASK3_DUAL_SETUP_YAML') or '').strip()
    yaml_payload, yaml_error = _task3_quick_parse_yaml(yaml_text)
    if yaml_payload:
        merged = _task3_quick_deep_update(merged, yaml_payload)
    return merged, yaml_text, yaml_error


TASK3_QUICK_INITIAL_SETUP, TASK3_QUICK_INITIAL_YAML_TEXT, TASK3_QUICK_INITIAL_YAML_ERROR = _task3_quick_build_initial_setup()


def _task3_quick_apply_smoke_env(setup):
    if os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE') != '1':
        return setup

    smoke_query = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_QUERY') or 'offline dual local model blind review').strip()
    smoke_processed = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_PROCESSED_DIR') or '').strip()
    smoke_out_dir = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_OUT_DIR') or 'runs/task3_dual_local_blind_ab_smoke').strip()
    smoke_model_a = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_MODEL_A') or 'mock-local-model-a').strip() or 'mock-local-model-a'
    smoke_model_b = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_MODEL_B') or 'mock-local-model-b').strip() or 'mock-local-model-b'

    merged = _task3_quick_deep_update(_task3_quick_default_setup(), setup or {})
    merged.update({
        'input_mode': 'query',
        'query': smoke_query,
        'processed_path': smoke_processed,
        'out_dir': smoke_out_dir,
        'search_limit': 5,
        'top_papers': 0,
        'top_hypotheses': 3,
        'candidate_top_k': 5,
        'top_pairs': 3,
        'include_multimodal': True,
        'run_vlm': False,
        'edge_mode': 'cooccurrence',
        'link_prediction_backend': 'heuristic',
        'hf_offline': True,
        'create_offline_form': True,
        'create_expert_bundle': True,
        'auto_download_offline': False,
        'auto_download_bundle': False,
        'auto_download_owner_key': False,
        'expert': {
            'last_name': os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_EXPERT_LAST', 'Smoke'),
            'first_name': os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_EXPERT_FIRST', 'Validation'),
            'patronymic': os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_EXPERT_PAT', '-'),
        },
        'domain_id': os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_DOMAIN_ID', 'science'),
    })
    merged['model_a'] = _task3_quick_deep_update(merged.get('model_a') or {}, {
        'owner_label': os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_OWNER_A', 'base_local_model'),
        'vlm_backend': 'none',
        'vlm_model_id': smoke_model_a,
        'local_text_model': os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_TEXT_A', ''),
    })
    merged['model_b'] = _task3_quick_deep_update(merged.get('model_b') or {}, {
        'owner_label': os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_OWNER_B', 'finetuned_local_model'),
        'vlm_backend': 'none',
        'vlm_model_id': smoke_model_b,
        'local_text_model': os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_TEXT_B', ''),
    })
    return merged


TASK3_QUICK_INITIAL_SETUP = _task3_quick_apply_smoke_env(TASK3_QUICK_INITIAL_SETUP)

input_mode = W.Dropdown(
    options=[
        ('Task 1 YAML', 'yaml'),
        ('Query + identifiers', 'query'),
        ('Commands', 'commands'),
    ],
    value=TASK3_QUICK_INITIAL_SETUP['input_mode'],
    description='Input mode',
)

task1_yaml_path = W.Text(
    value=TASK3_QUICK_INITIAL_SETUP['task1_yaml_path'],
    description='YAML path',
    placeholder='/content/task1.yaml',
    layout=W.Layout(width='100%'),
)
task1_yaml_upload = W.FileUpload(accept='.yaml,.yml', multiple=False, description='Upload YAML')

processed_path = W.Text(
    value=TASK3_QUICK_INITIAL_SETUP['processed_path'],
    description='Processed',
    placeholder='/content/processed_papers.zip или /content/processed_papers',
    layout=W.Layout(width='100%'),
)
processed_upload = W.FileUpload(accept='.zip', multiple=False, description='Upload ZIP')

commands_file_path = W.Text(
    value=TASK3_QUICK_INITIAL_SETUP['commands_file_path'],
    description='Commands path',
    placeholder='/content/commands.yaml',
    layout=W.Layout(width='100%'),
)
commands_upload = W.FileUpload(accept='.txt,.yaml,.yml,.json', multiple=False, description='Upload cmds')

query = W.Textarea(
    value=TASK3_QUICK_INITIAL_SETUP['query'],
    description='Query',
    placeholder='temporal knowledge graph multimodal hypothesis generation',
    layout=W.Layout(width='100%', height='90px'),
)
identifiers_text = W.Textarea(
    value='\n'.join(TASK3_QUICK_INITIAL_SETUP['identifiers']),
    description='IDs',
    placeholder='DOI / URL / identifier — по одному на строку',
    layout=W.Layout(width='100%', height='120px'),
)
commands_text = W.Textarea(
    value=TASK3_QUICK_INITIAL_SETUP['commands_text'],
    description='Commands',
    placeholder='query: ...\nidentifier: ...\nmodel_a_vlm_model_id: /models/base\nmodel_b_vlm_model_id: /models/tuned',
    layout=W.Layout(width='100%', height='150px'),
)

expert_last = W.Text(value=TASK3_QUICK_INITIAL_SETUP['expert']['last_name'], description='Фамилия', layout=W.Layout(width='100%'))
expert_first = W.Text(value=TASK3_QUICK_INITIAL_SETUP['expert']['first_name'], description='Имя', layout=W.Layout(width='100%'))
expert_pat = W.Text(value=TASK3_QUICK_INITIAL_SETUP['expert']['patronymic'], description='Отчество', layout=W.Layout(width='100%'))

domain_id = W.Text(value=TASK3_QUICK_INITIAL_SETUP['domain_id'], description='Domain', layout=W.Layout(width='100%'))
out_dir = W.Text(value=TASK3_QUICK_INITIAL_SETUP['out_dir'], description='Out dir', layout=W.Layout(width='100%'))

model_a_owner_label = W.Text(value=TASK3_QUICK_INITIAL_SETUP['model_a']['owner_label'], description='Owner α', layout=W.Layout(width='100%'))
model_a_vlm_backend = W.Dropdown(
    options=['qwen2_vl', 'qwen3_vl', 'llava', 'phi3_vision', 'auto', 'none'],
    value=TASK3_QUICK_INITIAL_SETUP['model_a']['vlm_backend'],
    description='VLM α',
)
model_a_vlm_model_id = W.Text(
    value=TASK3_QUICK_INITIAL_SETUP['model_a']['vlm_model_id'],
    description='Model/path α',
    placeholder='Qwen/Qwen2.5-VL-3B-Instruct или /models/base-vlm',
    layout=W.Layout(width='100%'),
)
model_a_local_text_model = W.Text(
    value=TASK3_QUICK_INITIAL_SETUP['model_a']['local_text_model'],
    description='Text α',
    placeholder='необязательно',
    layout=W.Layout(width='100%'),
)

model_b_owner_label = W.Text(value=TASK3_QUICK_INITIAL_SETUP['model_b']['owner_label'], description='Owner β', layout=W.Layout(width='100%'))
model_b_vlm_backend = W.Dropdown(
    options=['qwen2_vl', 'qwen3_vl', 'llava', 'phi3_vision', 'auto', 'none'],
    value=TASK3_QUICK_INITIAL_SETUP['model_b']['vlm_backend'],
    description='VLM β',
)
model_b_vlm_model_id = W.Text(
    value=TASK3_QUICK_INITIAL_SETUP['model_b']['vlm_model_id'],
    description='Model/path β',
    placeholder='/models/finetuned-vlm',
    layout=W.Layout(width='100%'),
)
model_b_local_text_model = W.Text(
    value=TASK3_QUICK_INITIAL_SETUP['model_b']['local_text_model'],
    description='Text β',
    placeholder='необязательно',
    layout=W.Layout(width='100%'),
)

hf_offline = W.Checkbox(value=bool(TASK3_QUICK_INITIAL_SETUP['hf_offline']), description='HF offline / local-files mode')
include_multimodal = W.Checkbox(value=bool(TASK3_QUICK_INITIAL_SETUP['include_multimodal']), description='Include multimodal')
run_vlm = W.Checkbox(value=bool(TASK3_QUICK_INITIAL_SETUP['run_vlm']), description='Run VLM')
raw_yaml_override = W.Textarea(
    value=TASK3_QUICK_INITIAL_YAML_TEXT,
    description='YAML override',
    placeholder='Необязательно: точечное YAML-переопределение',
    layout=W.Layout(width='100%', height='120px'),
)

quick_apply_btn = W.Button(description='Сохранить быстрый сетап', button_style='info')
quick_reset_btn = W.Button(description='Сбросить к defaults', button_style='')
quick_status_html = W.HTML()
quick_summary_html = W.HTML()


def _task3_quick_snapshot_fingerprint(snapshot):
    try:
        normalized = json.dumps(snapshot, ensure_ascii=False, sort_keys=True, default=str)
    except Exception:
        normalized = repr(snapshot)
    return hashlib.sha256(normalized.encode('utf-8')).hexdigest()


def _task3_quick_invalidate_validation(reason=''):
    globals()['TASK3_QUICK_SETUP_VALIDATED_OK'] = False
    globals()['TASK3_QUICK_SETUP_VALIDATED_FINGERPRINT'] = ''
    globals()['TASK3_QUICK_SETUP_VALIDATION_REQUIRED'] = True
    globals()['TASK3_QUICK_SETUP_DIRTY'] = True
    if reason:
        globals()['TASK3_QUICK_SETUP_BLOCK_REASON'] = reason

def _task3_quick_collect_snapshot():
    yaml_upload_path = ''
    processed_upload_path = ''
    commands_upload_path = ''
    upload_notes = []

    try:
        if task1_yaml_upload.value:
            yaml_upload_path = _task3_quick_save_upload(task1_yaml_upload.value, 'task1.yaml')
            upload_notes.append(f'Task1 YAML сохранён: {yaml_upload_path}')
    except Exception as e:
        upload_notes.append(f'Не удалось сохранить Task1 YAML upload: {type(e).__name__}: {e}')

    try:
        if processed_upload.value:
            processed_upload_path = _task3_quick_save_upload(processed_upload.value, 'processed_papers.zip')
            upload_notes.append(f'Processed ZIP сохранён: {processed_upload_path}')
    except Exception as e:
        upload_notes.append(f'Не удалось сохранить processed upload: {type(e).__name__}: {e}')

    try:
        if commands_upload.value:
            commands_upload_path = _task3_quick_save_upload(commands_upload.value, 'commands.yaml')
            upload_notes.append(f'Commands file сохранён: {commands_upload_path}')
    except Exception as e:
        upload_notes.append(f'Не удалось сохранить commands upload: {type(e).__name__}: {e}')

    return {
        'input_mode': input_mode.value,
        'task1_yaml_path': (yaml_upload_path or task1_yaml_path.value or '').strip(),
        'processed_path': (processed_upload_path or processed_path.value or '').strip(),
        'commands_file_path': (commands_upload_path or commands_file_path.value or '').strip(),
        'query': query.value.strip(),
        'identifiers': [line.strip() for line in identifiers_text.value.splitlines() if line.strip()],
        'commands_text': commands_text.value.strip(),
        'expert': {
            'last_name': expert_last.value.strip(),
            'first_name': expert_first.value.strip(),
            'patronymic': expert_pat.value.strip() or '-',
        },
        'domain_id': domain_id.value.strip() or 'science',
        'out_dir': out_dir.value.strip() or 'runs/task3_dual_local_blind_ab',
        'hf_offline': bool(hf_offline.value),
        'include_multimodal': bool(include_multimodal.value),
        'run_vlm': bool(run_vlm.value),
        'model_a': {
            'owner_label': model_a_owner_label.value.strip() or 'base_local_model',
            'vlm_backend': model_a_vlm_backend.value,
            'vlm_model_id': model_a_vlm_model_id.value.strip(),
            'local_text_model': model_a_local_text_model.value.strip(),
        },
        'model_b': {
            'owner_label': model_b_owner_label.value.strip() or 'finetuned_local_model',
            'vlm_backend': model_b_vlm_backend.value,
            'vlm_model_id': model_b_vlm_model_id.value.strip(),
            'local_text_model': model_b_local_text_model.value.strip(),
        },
        'yaml_override_text': raw_yaml_override.value,
        'upload_notes': upload_notes,
    }


def _task3_quick_apply_to_globals(show_message=True):
    snapshot = _task3_quick_collect_snapshot()
    merged = _task3_quick_deep_update(_task3_quick_default_setup(), snapshot)
    merged.pop('yaml_override_text', None)
    merged.pop('upload_notes', None)

    yaml_text = str(snapshot.get('yaml_override_text') or '').strip()
    yaml_payload, yaml_error = _task3_quick_parse_yaml(yaml_text)
    if yaml_payload:
        merged = _task3_quick_deep_update(merged, yaml_payload)

    fingerprint = _task3_quick_snapshot_fingerprint(snapshot)

    globals()['TASK3_DUAL_SETUP'] = merged
    globals()['TASK3_DUAL_SETUP_YAML'] = yaml_text
    globals()['TASK3_QUICK_LAST_SNAPSHOT'] = snapshot
    globals()['TASK3_QUICK_LAST_YAML_ERROR'] = yaml_error
    globals()['TASK3_QUICK_SETUP_CURRENT_FINGERPRINT'] = fingerprint
    globals()['TASK3_QUICK_SETUP_DIRTY'] = False
    globals()['TASK3_QUICK_SETUP_VALIDATED_OK'] = False
    globals()['TASK3_QUICK_SETUP_VALIDATED_FINGERPRINT'] = ''
    globals()['TASK3_QUICK_SETUP_VALIDATION_REQUIRED'] = True
    globals()['TASK3_QUICK_SETUP_BLOCK_REASON'] = 'После сохранения быстрый сетап нужно проверить отдельной ячейкой.'

    summary = {
        'input_mode': merged['input_mode'],
        'task1_yaml_path': merged['task1_yaml_path'],
        'processed_path': merged['processed_path'],
        'commands_file_path': merged['commands_file_path'],
        'query': merged['query'],
        'identifiers_count': len(merged['identifiers']),
        'model_a': merged['model_a'],
        'model_b': merged['model_b'],
        'hf_offline': merged['hf_offline'],
        'include_multimodal': merged['include_multimodal'],
        'run_vlm': merged['run_vlm'],
    }
    quick_summary_html.value = _task3_quick_render_box(
        'Что сохранено в быстрый сетап',
        lines=[
            f"input_mode = {merged['input_mode']}",
            f"identifiers = {len(merged['identifiers'])}",
            f"model/path α = {merged['model_a']['vlm_model_id'] or '—'}",
            f"model/path β = {merged['model_b']['vlm_model_id'] or '—'}",
        ],
        tone='neutral',
        details=json.dumps(summary, ensure_ascii=False, indent=2),
    )

    if show_message:
        lines = list(snapshot.get('upload_notes') or [])
        if yaml_error:
            lines.append(f'YAML override сохранён, но пока не разобран: {yaml_error}')
            tone = 'warning'
        else:
            lines.append('Значения сохранены в TASK3_DUAL_SETUP и будут подхвачены ячейкой установки.')
            tone = 'success'
        quick_status_html.value = _task3_quick_render_box('Быстрый сетап сохранён', lines=lines, tone=tone)

    return snapshot


def _task3_quick_reset_to_defaults(_=None):
    defaults = _task3_quick_default_setup()
    input_mode.value = defaults['input_mode']
    task1_yaml_path.value = defaults['task1_yaml_path']
    processed_path.value = defaults['processed_path']
    commands_file_path.value = defaults['commands_file_path']
    query.value = defaults['query']
    identifiers_text.value = '\n'.join(defaults['identifiers'])
    commands_text.value = defaults['commands_text']
    expert_last.value = defaults['expert']['last_name']
    expert_first.value = defaults['expert']['first_name']
    expert_pat.value = defaults['expert']['patronymic']
    domain_id.value = defaults['domain_id']
    out_dir.value = defaults['out_dir']
    model_a_owner_label.value = defaults['model_a']['owner_label']
    model_a_vlm_backend.value = defaults['model_a']['vlm_backend']
    model_a_vlm_model_id.value = defaults['model_a']['vlm_model_id']
    model_a_local_text_model.value = defaults['model_a']['local_text_model']
    model_b_owner_label.value = defaults['model_b']['owner_label']
    model_b_vlm_backend.value = defaults['model_b']['vlm_backend']
    model_b_vlm_model_id.value = defaults['model_b']['vlm_model_id']
    model_b_local_text_model.value = defaults['model_b']['local_text_model']
    hf_offline.value = defaults['hf_offline']
    include_multimodal.value = defaults['include_multimodal']
    run_vlm.value = defaults['run_vlm']
    raw_yaml_override.value = ''
    globals()['TASK3_QUICK_SETUP_DIRTY'] = True
    globals()['TASK3_QUICK_SETUP_VALIDATED_OK'] = False
    globals()['TASK3_QUICK_SETUP_VALIDATED_FINGERPRINT'] = ''
    globals()['TASK3_QUICK_SETUP_VALIDATION_REQUIRED'] = True
    globals()['TASK3_QUICK_SETUP_BLOCK_REASON'] = 'Поля были изменены. Сохраните быстрый сетап и заново запустите проверку.'
    quick_status_html.value = _task3_quick_render_box('Поля сброшены', lines=['Можно заново заполнить быстрый сетап. После изменений снова сохраните и проверьте сетап.'], tone='neutral')
    quick_summary_html.value = ''


def _task3_quick_on_apply(_):
    _task3_quick_apply_to_globals(show_message=True)


quick_apply_btn.on_click(_task3_quick_on_apply)
quick_reset_btn.on_click(_task3_quick_reset_to_defaults)


def _task3_quick_mark_dirty(change=None):
    if change is not None and change.get('name') not in (None, 'value'):
        return
    _task3_quick_invalidate_validation('Поля сетапа изменились после последней проверки.')
    quick_status_html.value = _task3_quick_render_box(
        'Сетап изменён',
        lines=['После изменений снова нажмите «Сохранить быстрый сетап», затем запустите отдельную ячейку проверки.'],
        tone='warning',
    )


for _widget in [
    input_mode,
    task1_yaml_path,
    task1_yaml_upload,
    processed_path,
    processed_upload,
    commands_file_path,
    commands_upload,
    query,
    identifiers_text,
    commands_text,
    expert_last,
    expert_first,
    expert_pat,
    domain_id,
    out_dir,
    model_a_owner_label,
    model_a_vlm_backend,
    model_a_vlm_model_id,
    model_a_local_text_model,
    model_b_owner_label,
    model_b_vlm_backend,
    model_b_vlm_model_id,
    model_b_local_text_model,
    hf_offline,
    include_multimodal,
    run_vlm,
    raw_yaml_override,
]:
    try:
        _widget.observe(_task3_quick_mark_dirty, names='value')
    except Exception:
        pass

TASK3_QUICK_WIDGETS = {
    'input_mode': input_mode,
    'task1_yaml_path': task1_yaml_path,
    'task1_yaml_upload': task1_yaml_upload,
    'processed_path': processed_path,
    'processed_upload': processed_upload,
    'commands_file_path': commands_file_path,
    'commands_upload': commands_upload,
    'query': query,
    'identifiers_text': identifiers_text,
    'commands_text': commands_text,
    'expert_last': expert_last,
    'expert_first': expert_first,
    'expert_pat': expert_pat,
    'domain_id': domain_id,
    'out_dir': out_dir,
    'model_a_owner_label': model_a_owner_label,
    'model_a_vlm_backend': model_a_vlm_backend,
    'model_a_vlm_model_id': model_a_vlm_model_id,
    'model_a_local_text_model': model_a_local_text_model,
    'model_b_owner_label': model_b_owner_label,
    'model_b_vlm_backend': model_b_vlm_backend,
    'model_b_vlm_model_id': model_b_vlm_model_id,
    'model_b_local_text_model': model_b_local_text_model,
    'hf_offline': hf_offline,
    'include_multimodal': include_multimodal,
    'run_vlm': run_vlm,
    'raw_yaml_override': raw_yaml_override,
    'apply_btn': quick_apply_btn,
    'reset_btn': quick_reset_btn,
    'status_html': quick_status_html,
    'summary_html': quick_summary_html,
}

basic_box = W.VBox([
    input_mode,
    W.HBox([task1_yaml_path, task1_yaml_upload]),
    W.HBox([processed_path, processed_upload]),
    W.HBox([commands_file_path, commands_upload]),
    query,
    identifiers_text,
    commands_text,
])

expert_box = W.VBox([expert_last, expert_first, expert_pat, domain_id, out_dir])
model_alpha_box = W.VBox([model_a_owner_label, model_a_vlm_backend, model_a_vlm_model_id, model_a_local_text_model])
model_beta_box = W.VBox([model_b_owner_label, model_b_vlm_backend, model_b_vlm_model_id, model_b_local_text_model])
flags_box = W.VBox([hf_offline, include_multimodal, run_vlm, raw_yaml_override])

accordion = W.Accordion(children=[
    basic_box,
    W.HBox([expert_box, flags_box]),
    W.HBox([model_alpha_box, model_beta_box]),
])
accordion.set_title(0, 'Входные данные')
accordion.set_title(1, 'Эксперт и флаги')
accordion.set_title(2, 'Модели α / β')

display(W.VBox([
    W.HTML('<div style="font-weight:700; font-size:16px; margin-bottom:4px;">Быстрый сетап до установки репозитория</div>'
           '<div style="color:#475569; margin-bottom:8px;">Эта ячейка нужна только для раннего ввода данных. Проверка вынесена в следующую отдельную ячейку.</div>'),
    accordion,
    W.HBox([quick_apply_btn, quick_reset_btn]),
    quick_status_html,
    quick_summary_html,
]))

_task3_quick_apply_to_globals(show_message=False)
if TASK3_QUICK_INITIAL_YAML_ERROR:
    quick_status_html.value = _task3_quick_render_box(
        'Есть проблема в сохранённом YAML override',
        lines=[TASK3_QUICK_INITIAL_YAML_ERROR],
        tone='warning',
    )


In [5]:
# @title
# Проверка быстрого сетапа до установки репозитория
# Эта ячейка отдельно проверяет ранний графический конфиг и показывает, что нужно исправить.

from pathlib import Path
import json

try:
    import yaml
except Exception:
    yaml = None

try:
    import ipywidgets as W
    from IPython.display import display
except Exception as e:
    raise RuntimeError(f'ipywidgets недоступен в окружении: {type(e).__name__}: {e}')


def _task3_quick_validate_snapshot(snapshot):
    errors = []
    warnings = []

    input_mode = str(snapshot.get('input_mode') or 'query').strip()
    task1_yaml_path = str(snapshot.get('task1_yaml_path') or '').strip()
    processed_path = str(snapshot.get('processed_path') or '').strip()
    commands_file_path = str(snapshot.get('commands_file_path') or '').strip()
    query_text = str(snapshot.get('query') or '').strip()
    commands_text = str(snapshot.get('commands_text') or '').strip()
    identifiers = [str(x).strip() for x in (snapshot.get('identifiers') or []) if str(x).strip()]
    yaml_override_text = str(snapshot.get('yaml_override_text') or '').strip()

    if input_mode == 'yaml' and not task1_yaml_path:
        errors.append('Для режима "Task 1 YAML" нужно указать YAML path или загрузить YAML файл.')
    if input_mode == 'commands' and not (commands_file_path or commands_text):
        errors.append('Для режима "Commands" нужен commands path, upload или текст commands.')
    if input_mode == 'query' and not (query_text or identifiers or processed_path):
        errors.append('Для режима "Query + identifiers" укажите query, identifiers, либо processed path.')

    for label, path_value, must_exist in [
        ('Task 1 YAML', task1_yaml_path, bool(task1_yaml_path)),
        ('Processed path', processed_path, bool(processed_path)),
        ('Commands path', commands_file_path, bool(commands_file_path)),
    ]:
        if must_exist:
            p = Path(path_value).expanduser()
            if not p.exists():
                errors.append(f'{label}: путь не найден → {path_value}')

    if yaml_override_text:
        if yaml is None:
            warnings.append('PyYAML недоступен: YAML override не удалось проверить в этой ячейке.')
        else:
            try:
                payload = yaml.safe_load(yaml_override_text) or {}
                if not isinstance(payload, dict):
                    errors.append('YAML override должен содержать объект верхнего уровня.')
            except Exception as e:
                errors.append(f'YAML override не разобран: {type(e).__name__}: {e}')

    model_a = snapshot.get('model_a') or {}
    model_b = snapshot.get('model_b') or {}

    if snapshot.get('run_vlm', True):
        if str(model_a.get('vlm_backend') or '').strip().lower() != 'none' and not str(model_a.get('vlm_model_id') or '').strip():
            warnings.append('Для α не указан model/path. Если модель локальная, задайте путь или model id заранее.')
        if str(model_b.get('vlm_backend') or '').strip().lower() != 'none' and not str(model_b.get('vlm_model_id') or '').strip():
            warnings.append('Для β не указан model/path. Если модель локальная, задайте путь или model id заранее.')

    if str(model_a.get('owner_label') or '').strip() == str(model_b.get('owner_label') or '').strip():
        warnings.append('Owner labels α и β совпадают. Для удобства лучше сделать их разными.')

    if not str((snapshot.get('expert') or {}).get('last_name') or '').strip():
        warnings.append('Не указана фамилия эксперта.')
    if not str((snapshot.get('expert') or {}).get('first_name') or '').strip():
        warnings.append('Не указано имя эксперта.')


    out_dir = str(snapshot.get('out_dir') or '').strip()
    if Path('/kaggle/working').exists() and out_dir:
        try:
            resolved_out = Path(out_dir).expanduser()
            if not resolved_out.is_absolute():
                resolved_out = Path('/kaggle/working') / resolved_out
            resolved_out = resolved_out.resolve()
            kaggle_root = Path('/kaggle/working').resolve()
            if kaggle_root not in [resolved_out, *resolved_out.parents]:
                warnings.append('На Kaggle лучше писать out_dir внутрь /kaggle/working, иначе выходные файлы версии могут быть неудобны для скачивания.')
        except Exception:
            warnings.append('Не удалось проверить out_dir для Kaggle.')

    summary = {
        'input_mode': input_mode,
        'task1_yaml_path': task1_yaml_path,
        'processed_path': processed_path,
        'commands_file_path': commands_file_path,
        'query_present': bool(query_text),
        'identifiers_count': len(identifiers),
        'commands_text_present': bool(commands_text),
        'model_a': model_a,
        'model_b': model_b,
        'hf_offline': bool(snapshot.get('hf_offline')),
        'run_vlm': bool(snapshot.get('run_vlm')),
    }
    return {'ok': not errors, 'errors': errors, 'warnings': warnings, 'summary': summary}



class Task3SetupBlocked(RuntimeError):
    def __init__(self, title, lines=None):
        self.title = str(title)
        self.lines = [str(x) for x in (lines or []) if str(x).strip()]
        message = self.title
        if self.lines:
            message += '\n- ' + '\n- '.join(self.lines)
        super().__init__(message)

    def _render_traceback_(self):
        try:
            display(W.HTML(_task3_quick_render_box(self.title, lines=self.lines, tone='warning')))
            return []
        except Exception:
            return super()._render_traceback_()


def _task3_require_validated_setup():
    if globals().get('TASK3_QUICK_SETUP_DIRTY'):
        raise Task3SetupBlocked(
            'Запуск заблокирован: сетап изменён после последней проверки',
            [
                'Нажмите «Сохранить быстрый сетап».',
                'Затем снова запустите ячейку «Проверка быстрого сетапа».',
            ],
        )

    report = globals().get('TASK3_QUICK_VALIDATION_REPORT')
    if not report:
        raise Task3SetupBlocked(
            'Запуск заблокирован: быстрый сетап ещё не проверен',
            [
                'Сначала запустите ячейку «Проверка быстрого сетапа».',
            ],
        )

    if not globals().get('TASK3_QUICK_SETUP_VALIDATED_OK'):
        lines = list((report or {}).get('errors') or [])
        reason = globals().get('TASK3_QUICK_SETUP_BLOCK_REASON')
        if reason:
            lines = [reason] + lines
        if not lines:
            lines = ['Исправьте конфиг и заново запустите ячейку проверки.']
        raise Task3SetupBlocked('Запуск заблокирован: быстрый сетап не прошёл проверку', lines[:8])

    current_fp = str(globals().get('TASK3_QUICK_SETUP_CURRENT_FINGERPRINT') or '')
    validated_fp = str(globals().get('TASK3_QUICK_SETUP_VALIDATED_FINGERPRINT') or '')
    if current_fp and validated_fp and current_fp != validated_fp:
        raise Task3SetupBlocked(
            'Запуск заблокирован: проверка устарела',
            [
                'Сохранённая конфигурация изменилась после последней проверки.',
                'Снова запустите ячейку «Проверка быстрого сетапа».',
            ],
        )

    return True


quick_validate_btn = W.Button(description='Проверить быстрый сетап', button_style='success')
quick_validation_html = W.HTML()
quick_validation_summary = W.HTML()

def _task3_quick_run_validation(_=None):
    if '_task3_quick_apply_to_globals' not in globals():
        quick_validation_html.value = _task3_quick_render_box(
            'Быстрый сетап ещё не создан',
            lines=['Сначала запустите предыдущую ячейку с графическим меню.'],
            tone='warning',
        )
        quick_validation_summary.value = ''
        return

    snapshot = _task3_quick_apply_to_globals(show_message=False)
    report = _task3_quick_validate_snapshot(snapshot)
    fingerprint = globals().get('TASK3_QUICK_SETUP_CURRENT_FINGERPRINT') or _task3_quick_snapshot_fingerprint(snapshot)
    globals()['TASK3_QUICK_VALIDATION_REPORT'] = report
    globals()['TASK3_QUICK_SETUP_DIRTY'] = False
    globals()['TASK3_QUICK_SETUP_VALIDATION_REQUIRED'] = not report['ok']
    globals()['TASK3_QUICK_SETUP_VALIDATED_OK'] = bool(report['ok'])
    globals()['TASK3_QUICK_SETUP_VALIDATED_FINGERPRINT'] = fingerprint if report['ok'] else ''
    globals()['TASK3_QUICK_SETUP_BLOCK_REASON'] = '' if report['ok'] else 'Исправьте ошибки в быстром сетапе и снова запустите проверку.'

    if report['ok']:
        title = 'Быстрый сетап выглядит корректно'
        tone = 'success'
        lines = ['Проверка пройдена. Ячейки установки, формы и запуска теперь разблокированы.']
    else:
        title = 'В быстром сетапе есть ошибки'
        tone = 'danger'
        lines = list(report['errors']) + ['Пока ошибки не исправлены, остальные рабочие ячейки будут заблокированы.']

    if report['warnings']:
        lines = list(lines) + [f'Предупреждения: {len(report["warnings"])}']

    details = json.dumps(report['summary'], ensure_ascii=False, indent=2)
    quick_validation_html.value = _task3_quick_render_box(title, lines=lines, tone=tone, details=details)

    if report['warnings']:
        quick_validation_summary.value = _task3_quick_render_box(
            'Что стоит проверить дополнительно',
            lines=report['warnings'],
            tone='warning',
        )
    else:
        quick_validation_summary.value = ''

quick_validate_btn.on_click(_task3_quick_run_validation)

display(W.VBox([
    W.HTML('<div style="font-weight:700; font-size:16px; margin-bottom:4px;">Отдельная проверка быстрого сетапа</div>'
           '<div style="color:#475569; margin-bottom:8px;">Эта ячейка ничего не запускает. Она только валидирует ранний конфиг и подсказывает, что исправить.</div>'),
    quick_validate_btn,
    quick_validation_html,
    quick_validation_summary,
]))

_task3_quick_run_validation()


In [6]:
# @title
# Установка и импорт (запустите эту ячейку)

if '_task3_require_validated_setup' not in globals():
    raise RuntimeError('Сначала запустите ячейку быстрого сетапа и отдельную ячейку проверки.')
_task3_require_validated_setup()

import gc, io, json, os, sys, tempfile, subprocess, zipfile, shutil, traceback, inspect
from pathlib import Path
import importlib.util

os.environ.setdefault('G4F_ASYNC_ENABLED', '1')
os.environ.setdefault('G4F_ASYNC_MAX_CONCURRENCY', '3')
os.environ.setdefault('G4F_ASYNC_RETRIES', '3')
os.environ.setdefault('G4F_ASYNC_MAX_MODELS_PER_REQUEST', '3')
os.environ.setdefault('LLM_REQUEST_TIMEOUT_SECONDS', '25')
os.environ.setdefault('VLM_REQUEST_TIMEOUT_SECONDS', '45')


TASK3_IS_KAGGLE = Path('/kaggle/working').exists() or bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or os.environ.get('TASK3_NOTEBOOK_PLATFORM') == 'kaggle'
TASK3_RUNTIME_OFFLINE = bool(TASK3_IS_KAGGLE or os.environ.get('HF_HUB_OFFLINE') == '1' or os.environ.get('TRANSFORMERS_OFFLINE') == '1')


def _task3_module_available(module_name: str) -> bool:
    try:
        return importlib.util.find_spec(module_name) is not None
    except Exception:
        return False

if os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE') == '1':
    os.environ.setdefault('LLM_PROVIDER', 'mock')
    os.environ.setdefault('LLM_MODEL', 'mock')
    os.environ.setdefault('EMBED_PROVIDER', 'hash')
    os.environ.setdefault('MM_EMBED_BACKEND', 'none')
    os.environ.setdefault('VLM_BACKEND', 'none')
    os.environ.setdefault('HF_HUB_OFFLINE', '1')
    os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
    os.environ.setdefault('HF_DATASETS_OFFLINE', '1')


def run(cmd, cwd=None):
    print('>', ' '.join(map(str, cmd)))
    subprocess.check_call(cmd, cwd=cwd)


def run_optional(cmd, cwd=None, label=None):
    try:
        run(cmd, cwd=cwd)
        return True
    except Exception as e:
        label = label or ' '.join(map(str, cmd))
        print(f'[warn] optional step failed: {label}: {type(e).__name__}: {e}')
        return False


def _materialize_local_repo_archive() -> Path | None:
    search_roots = [Path('/mnt/data'), Path('/content'), Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd(), Path.cwd().parent]
    patterns = (
        'top-papers-graph-main-task3-module.zip',
        'top-papers-graph-main-task3-notebook.zip',
        'top-papers-graph-main.zip',
        'top-papers-graph*.zip',
        '*top-papers-graph*.zip',
    )
    for base in search_roots:
        if not base.exists():
            continue
        for pattern in patterns:
            archives = list(base.glob(pattern))
            if base.name == 'input':
                archives.extend(base.rglob(pattern))
            for archive in sorted({p.resolve() if p.exists() else p for p in archives}):
                target = archive.with_suffix('')
                try:
                    candidate_root = target / 'top-papers-graph-main'
                    if (candidate_root / 'pyproject.toml').exists() and (candidate_root / 'src' / 'scireason').exists():
                        return candidate_root
                    if target.exists() and (target / 'pyproject.toml').exists() and (target / 'src' / 'scireason').exists():
                        return target
                    target.mkdir(parents=True, exist_ok=True)
                    with zipfile.ZipFile(archive, 'r') as zf:
                        zf.extractall(target)
                    if (candidate_root / 'pyproject.toml').exists() and (candidate_root / 'src' / 'scireason').exists():
                        return candidate_root
                    if (target / 'pyproject.toml').exists() and (target / 'src' / 'scireason').exists():
                        return target
                except Exception as e:
                    print(f'[warn] local repo archive skipped: {archive}: {type(e).__name__}: {e}')
    return None


def find_repo_root() -> Path | None:
    env_dir = os.environ.get('TPG_REPO_DIR')
    if env_dir:
        env_path = Path(env_dir)
        if (env_path / 'pyproject.toml').exists() and (env_path / 'src' / 'scireason').exists():
            return env_path

    candidates = []
    repo_from_archive = _materialize_local_repo_archive()
    if repo_from_archive is not None:
        candidates.append(repo_from_archive)
    if env_dir:
        candidates.append(Path(env_dir))
    cwd = Path.cwd()
    candidates.extend([cwd, cwd.parent, Path('/content'), Path('/mnt/data'), Path('/kaggle/input'), Path('/kaggle/working')])
    search_bases = [Path('/content'), Path('/mnt/data'), Path('/kaggle/input'), Path('/kaggle/working'), cwd, cwd.parent]
    patterns = ('top-papers-graph-main*', 'top-papers-graph*')
    for base in search_bases:
        if not base.exists():
            continue
        for pattern in patterns:
            candidates.extend(base.glob(pattern))
            if base.name == 'input':
                candidates.extend(base.rglob(pattern))
    seen = set()
    uniq = []
    for c in candidates:
        try:
            r = c.resolve()
        except Exception:
            r = c
        key = str(r)
        if key in seen:
            continue
        seen.add(key)
        uniq.append(r)
    for c in uniq:
        if (c / 'pyproject.toml').exists() and (c / 'src' / 'scireason').exists():
            return c
    return None


TASK3_DUAL_BOOT_OK = False
TASK3_DUAL_BOOT_ERROR = None
TASK3_DEFAULT_LOCAL_VLM_MODEL = os.environ.get('TASK3_DEFAULT_LOCAL_VLM_MODEL', 'Qwen/Qwen2.5-VL-3B-Instruct') or 'Qwen/Qwen2.5-VL-3B-Instruct'
TASK3_DUAL_PIPELINE_SUPPORTS_PROGRESS = False
TASK3_DUAL_LAST_RESULT = None
TASK3_DUAL_LAST_TASK_META = None
TASK3_DUAL_LAST_ARTIFACTS = None
TASK3_DUAL_LAST_ERROR = None

REPO_DIR = find_repo_root()
REPO_URL = 'https://github.com/top-papers/top-papers-graph.git'
if REPO_DIR is None:
    REPO_DIR = Path('/content/top-papers-graph') if Path('/content').exists() else (Path('/kaggle/working') / 'top-papers-graph' if TASK3_IS_KAGGLE else Path.cwd() / 'top-papers-graph')
    if not REPO_DIR.exists():
        allow_git_clone = os.environ.get('TASK3_ALLOW_GIT_CLONE', '0' if TASK3_IS_KAGGLE else '1') == '1'
        if allow_git_clone:
            run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
        else:
            raise FileNotFoundError(
                'Не найден локальный репозиторий task3. Для Kaggle offline прикрепите архив/датасет с top-papers-graph в /kaggle/input или задайте TPG_REPO_DIR.'
            )

SRC_DIR = REPO_DIR / 'src'
if not SRC_DIR.exists():
    raise FileNotFoundError(f'Не найден каталог src в репозитории: {REPO_DIR}')
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

_required_boot_modules = ['ipywidgets', 'yaml', 'requests', 'nbformat']
_missing_boot_modules = [name for name in _required_boot_modules if not _task3_module_available(name)]
if _missing_boot_modules and not TASK3_RUNTIME_OFFLINE:
    run_optional([sys.executable, '-m', 'pip', 'install', '-q', 'ipywidgets', 'pyyaml', 'requests', 'unidecode', 'nbformat'])
elif _missing_boot_modules:
    print('[warn] offline mode: пропускаю pip install для базовых notebook-зависимостей ->', ', '.join(_missing_boot_modules))

_task3_editable_install = [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[task3]']
if TASK3_RUNTIME_OFFLINE or os.environ.get('TASK3_NOTEBOOK_SMOKE') == '1' or os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE') == '1' or os.environ.get('TASK3_NOTEBOOK_SKIP_OPTIONAL_DEPS') == '1':
    _task3_editable_install = [sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps']
if not run_optional(_task3_editable_install, cwd=REPO_DIR, label='pip install editable task3 notebook runtime'):
    if _task3_editable_install[-1:] != ['--no-deps']:
        run_optional([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], cwd=REPO_DIR, label='pip install editable task3 notebook runtime --no-deps fallback')

prepare_task3_hypothesis_bundle = None
build_task3_dual_model_expert_bundle = None
build_task3_dual_model_offline_review_package = None

try:
    import yaml
    import ipywidgets as W
    from IPython.display import HTML, Markdown, display, clear_output
    from unidecode import unidecode

    from scireason.config import settings
    from scireason.pipeline.task3_hypothesis_generation import prepare_task3_hypothesis_bundle
    from scireason.task3_dual_model_review import (
        build_task3_dual_model_expert_bundle,
        build_task3_dual_model_offline_review_package,
    )

    TASK3_DEFAULT_LOCAL_VLM_MODEL = getattr(settings, 'vlm_model_id', TASK3_DEFAULT_LOCAL_VLM_MODEL) or TASK3_DEFAULT_LOCAL_VLM_MODEL
    TASK3_DUAL_PIPELINE_SUPPORTS_PROGRESS = 'progress_callback' in inspect.signature(prepare_task3_hypothesis_bundle).parameters
    TASK3_DUAL_BOOT_OK = True
except Exception as e:
    TASK3_DUAL_BOOT_ERROR = {
        'type': type(e).__name__,
        'message': str(e),
        'traceback': traceback.format_exc(limit=6),
    }
    print('[error] Импорт notebook runtime завершился с ошибкой.')
    print(TASK3_DUAL_BOOT_ERROR['type'] + ': ' + TASK3_DUAL_BOOT_ERROR['message'])

print('REPO_DIR =', REPO_DIR)
print('SRC_DIR  =', SRC_DIR)
print('task3 default local VLM model =', TASK3_DEFAULT_LOCAL_VLM_MODEL)
print('progress callback support =', TASK3_DUAL_PIPELINE_SUPPORTS_PROGRESS)
if not TASK3_DUAL_BOOT_OK:
    print('[warn] Notebook runtime загружен не полностью. Форму можно открыть и поправить конфиг, но запуск будет недоступен, пока не исправится ошибка импорта.')

> git clone --depth 1 https://github.com/top-papers/top-papers-graph.git /content/top-papers-graph
> /usr/bin/python3 -m pip install -q ipywidgets pyyaml requests unidecode nbformat
> /usr/bin/python3 -m pip install -q -e .[task3]


/content/top-papers-graph/src/scireason/llm.py:681: SyntaxWarning: invalid escape sequence '\/'
  JSON escapes because JSON only allows ``\" \\ \/ \b \f \n \r \t`` and


REPO_DIR = /content/top-papers-graph
SRC_DIR  = /content/top-papers-graph/src
task3 default local VLM model = auto
progress callback support = True


In [7]:
# @title
# Вспомогательные функции (не нужно редактировать)

if '_task3_require_validated_setup' not in globals():
    raise RuntimeError('Сначала запустите ячейку быстрого сетапа и отдельную ячейку проверки.')
_task3_require_validated_setup()

import hashlib
import json
import re
import traceback
from pathlib import Path


def _escape_html(value):
    return html_escape(str(value or ''))


def html_escape(text: str) -> str:
    return (
        str(text or '')
        .replace('&', '&amp;')
        .replace('<', '&lt;')
        .replace('>', '&gt;')
        .replace('"', '&quot;')
        .replace("'", '&#39;')
    )


def _slugify(text: str) -> str:
    value = unidecode(str(text or '')).lower()
    value = re.sub(r'[^a-z0-9\s_-]+', '', value)
    value = re.sub(r'\s+', '_', value).strip('_')
    return value or 'expert'


def _materialize_upload(upload_value, out_dir: Path, default_name: str) -> Path | None:
    if not upload_value:
        return None
    out_dir.mkdir(parents=True, exist_ok=True)
    if isinstance(upload_value, dict):
        if not upload_value:
            return None
        name, meta = next(iter(upload_value.items()))
        content = meta.get('content')
        if content is None:
            return None
        path = out_dir / (name or default_name)
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path
    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        content = meta.get('content')
        if content is None:
            return None
        name = meta.get('name') or default_name
        path = out_dir / name
        path.write_bytes(content if isinstance(content, (bytes, bytearray)) else bytes(content))
        return path
    raise TypeError('Неподдерживаемый формат upload value')


def _read_upload_name_and_bytes(upload_value):
    if not upload_value:
        return None, None
    if isinstance(upload_value, dict) and upload_value:
        name, meta = next(iter(upload_value.items()))
        return name or 'upload.bin', meta.get('content')
    if isinstance(upload_value, tuple) and upload_value:
        meta = upload_value[0]
        return meta.get('name') or 'upload.bin', meta.get('content')
    return None, None


def _parse_bool(value, default=None):
    if value is None:
        return default
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {'1', 'true', 'yes', 'y', 'on', 'да'}:
        return True
    if text in {'0', 'false', 'no', 'n', 'off', 'нет'}:
        return False
    return default


def _parse_identifier_blob(text: str):
    parts = []
    for raw_line in str(text or '').splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        for token in re.split(r'[,;]', line):
            token = token.strip()
            if token:
                parts.append(token)
    return parts


def _parse_commands_payload(text: str) -> dict:
    payload = {
        'query': '',
        'identifiers': [],
        'domain_id': None,
        'top_papers': None,
        'top_hypotheses': None,
        'candidate_top_k': None,
        'search_limit': None,
        'top_pairs': None,
        'edge_mode': None,
        'link_prediction_backend': None,
        'include_multimodal': None,
        'run_vlm': None,
        'hf_offline': None,
        'processed_dir': None,
        'out_dir': None,
        'model_a_owner_label': None,
        'model_a_vlm_backend': None,
        'model_a_vlm_model_id': None,
        'model_a_local_text_model': None,
        'model_b_owner_label': None,
        'model_b_vlm_backend': None,
        'model_b_vlm_model_id': None,
        'model_b_local_text_model': None,
    }
    text = str(text or '').strip()
    if not text:
        return payload

    for parser in (json.loads, yaml.safe_load):
        try:
            obj = parser(text)
            if isinstance(obj, dict):
                for key in list(payload.keys()):
                    if key == 'identifiers':
                        continue
                    if key in obj:
                        payload[key] = obj[key]
                if 'identifier' in obj:
                    payload['identifiers'].extend(_parse_identifier_blob(obj['identifier']))
                if 'identifiers' in obj:
                    if isinstance(obj['identifiers'], list):
                        payload['identifiers'].extend([str(x).strip() for x in obj['identifiers'] if str(x).strip()])
                    else:
                        payload['identifiers'].extend(_parse_identifier_blob(obj['identifiers']))
                return payload
        except Exception:
            pass

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.startswith('#'):
            continue
        if ':' not in line:
            payload['identifiers'].append(line)
            continue
        key, value = line.split(':', 1)
        key = key.strip().lower()
        value = value.strip()
        if key in {'query', 'topic'}:
            payload['query'] = value
        elif key in {'identifier', 'id', 'doi', 'url'}:
            payload['identifiers'].extend(_parse_identifier_blob(value))
        elif key in {'identifiers', 'ids', 'doi_list', 'url_list'}:
            payload['identifiers'].extend(_parse_identifier_blob(value))
        elif key in {'domain', 'domain_id'}:
            payload['domain_id'] = value
        elif key in {'top_papers', 'top_hypotheses', 'candidate_top_k', 'search_limit', 'top_pairs'}:
            try:
                payload[key] = int(value)
            except Exception:
                pass
        elif key in {'edge_mode', 'link_prediction_backend', 'processed_dir', 'out_dir',
                     'model_a_owner_label', 'model_a_vlm_backend', 'model_a_vlm_model_id', 'model_a_local_text_model',
                     'model_b_owner_label', 'model_b_vlm_backend', 'model_b_vlm_model_id', 'model_b_local_text_model'}:
            payload[key] = value
        elif key in {'include_multimodal', 'run_vlm', 'hf_offline'}:
            payload[key] = _parse_bool(value, default=None)
    return payload


def _merge_command_payloads(base_payload: dict, extra_payload: dict) -> dict:
    merged = dict(base_payload or {})
    merged.setdefault('identifiers', [])
    for key, value in (extra_payload or {}).items():
        if key == 'identifiers':
            merged['identifiers'].extend([x for x in value if x])
        elif value not in (None, '', []):
            merged[key] = value
    deduped = []
    seen = set()
    for item in merged.get('identifiers') or []:
        norm = str(item).strip()
        if not norm or norm in seen:
            continue
        seen.add(norm)
        deduped.append(norm)
    merged['identifiers'] = deduped
    return merged


def _merged_commands_from_sources(commands_text_value: str, commands_upload_value=None, commands_path_text: str = '') -> dict:
    merged = _parse_commands_payload(commands_text_value)
    upload_name, upload_bytes = _read_upload_name_and_bytes(commands_upload_value)
    if upload_bytes:
        text = upload_bytes.decode('utf-8', errors='replace') if isinstance(upload_bytes, (bytes, bytearray)) else bytes(upload_bytes).decode('utf-8', errors='replace')
        merged = _merge_command_payloads(merged, _parse_commands_payload(text))
    if str(commands_path_text or '').strip():
        path = Path(str(commands_path_text).strip()).expanduser()
        if not path.exists():
            raise FileNotFoundError(f'Не найден commands file: {path}')
        merged = _merge_command_payloads(merged, _parse_commands_payload(path.read_text(encoding='utf-8')))
    return merged


def _merged_commands_from_widgets(commands_text_value: str, commands_upload_value) -> dict:
    return _merged_commands_from_sources(commands_text_value, commands_upload_value, '')


def _load_yaml_if_present(path: Path | None) -> dict:
    if path is None or not path.exists():
        return {}
    data = yaml.safe_load(path.read_text(encoding='utf-8')) or {}
    if isinstance(data, dict):
        return data
    raise ValueError('Ожидался YAML с top-level object')


def _discover_processed_dir(root: Path) -> Path | None:
    candidates = []
    for p in [root, *root.rglob('*')]:
        if not p.is_dir():
            continue
        has_meta = any(child.is_file() and child.name == 'meta.json' for child in p.glob('*/meta.json'))
        has_chunks = any(child.is_file() and child.name == 'chunks.jsonl' for child in p.glob('*/chunks.jsonl'))
        if has_meta or has_chunks:
            candidates.append(p)
    if candidates:
        candidates.sort(key=lambda p: len(p.parts))
        return candidates[0]
    return None


def _extract_processed_zip(upload_value, workdir: Path) -> Path | None:
    archive_path = _materialize_upload(upload_value, workdir, 'processed_papers.zip')
    if archive_path is None:
        return None
    extract_dir = workdir / 'processed_from_zip'
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive_path, 'r') as zf:
        zf.extractall(extract_dir)
    found = _discover_processed_dir(extract_dir)
    if found is None:
        raise FileNotFoundError('Не удалось найти processed_papers внутри ZIP-архива')
    return found


def _resolve_existing_path(path_text: str) -> Path | None:
    text = str(path_text or '').strip()
    if not text:
        return None
    return Path(text).expanduser()


def _materialize_local_file(path_text: str, out_dir: Path, default_name: str) -> Path | None:
    src = _resolve_existing_path(path_text)
    if src is None:
        return None
    if not src.exists():
        raise FileNotFoundError(f'Не найден файл: {src}')
    out_dir.mkdir(parents=True, exist_ok=True)
    target = out_dir / (src.name or default_name)
    if src.resolve() != target.resolve():
        shutil.copy2(src, target)
    return target


def _extract_processed_source(upload_value, path_text: str, workdir: Path) -> Path | None:
    uploaded = _extract_processed_zip(upload_value, workdir)
    if uploaded is not None:
        return uploaded

    src = _resolve_existing_path(path_text)
    if src is None:
        return None
    if not src.exists():
        raise FileNotFoundError(f'Не найден processed path: {src}')
    if src.is_dir():
        found = _discover_processed_dir(src)
        if found is None:
            raise FileNotFoundError(f'В директории не найден processed_papers: {src}')
        return found
    if src.suffix.lower() != '.zip':
        raise ValueError(f'Processed path должен указывать на zip или директорию, а не на {src}')
    extract_dir = workdir / 'processed_from_path'
    extract_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(src, 'r') as zf:
        zf.extractall(extract_dir)
    found = _discover_processed_dir(extract_dir)
    if found is None:
        raise FileNotFoundError('Не удалось найти processed_papers внутри ZIP по указанному пути')
    return found


def _task3_task_meta_from_widgets(base_doc: dict | None = None):
    base_doc = dict(base_doc or {})
    expert = base_doc.get('expert') if isinstance(base_doc.get('expert'), dict) else {}
    last_name = str(expert_last.value or expert.get('last_name') or '').strip()
    first_name = str(expert_first.value or expert.get('first_name') or '').strip()
    patronymic = str(expert_pat.value or expert.get('patronymic') or '-').strip() or '-'
    full_name = ' '.join(x for x in [last_name, first_name, patronymic] if x).strip()
    latin_slug = _slugify(full_name)

    query_candidate = str(query.value or '').strip()
    submission_id = str(base_doc.get('submission_id') or '').strip()
    if not submission_id:
        seed_text = query_candidate or str(base_doc.get('topic') or 'task3_dual_local')
        short_hash = hashlib.sha256(seed_text.encode('utf-8')).hexdigest()[:12]
        submission_id = f'{latin_slug or "expert"}__{short_hash}'

    topic_value = str(base_doc.get('topic') or query_candidate or '').strip()
    return {
        'topic': topic_value,
        'submission_id': submission_id,
        'cutoff_year': str(base_doc.get('cutoff_year') or '').strip(),
        'domain': str(base_doc.get('domain') or domain_id.value or '').strip(),
        'expert': {
            'last_name': last_name,
            'first_name': first_name,
            'patronymic': patronymic,
            'full_name': full_name,
            'latin_slug': latin_slug,
        },
    }



def _task3_is_kaggle_runtime() -> bool:
    return Path('/kaggle/working').exists() or bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE')) or os.environ.get('TASK3_NOTEBOOK_PLATFORM') == 'kaggle'


def _task3_default_out_dir() -> Path:
    if _task3_is_kaggle_runtime():
        return (Path('/kaggle/working') / 'task3_dual_local_blind_ab').resolve()
    return Path('runs/task3_dual_local_blind_ab')


def _task3_resolve_output_dir(path_text: str | None) -> Path:
    raw = str(path_text or '').strip()
    if not raw:
        return _task3_default_out_dir()
    candidate = Path(raw).expanduser()
    if _task3_is_kaggle_runtime() and not candidate.is_absolute():
        candidate = Path('/kaggle/working') / candidate
    return candidate.resolve() if candidate.is_absolute() else candidate


def _task3_write_run_manifest(out_dir: Path, payload: dict) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = out_dir / 'task3_dual_run_manifest.json'
    manifest_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    return manifest_path


def _task3_build_export_zip(out_dir: Path) -> Path | None:
    try:
        export_zip = out_dir.parent / f'{out_dir.name}__kaggle_outputs.zip'
        with zipfile.ZipFile(export_zip, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
            for path in sorted(out_dir.rglob('*')):
                if path.is_dir():
                    continue
                zf.write(path, arcname=str(path.relative_to(out_dir.parent)))
        return export_zip
    except Exception:
        return None

def _download_file_if_possible(path: Path, status_widget=None):
    try:
        from google.colab import files as colab_files
        colab_files.download(str(path))
        return True
    except Exception:
        if status_widget is not None:
            current = str(getattr(status_widget, 'value', '') or '')
            status_widget.value = current + f'<div><code>{_escape_html(str(path))}</code></div>'
        return False


def _top_hypotheses_summary(hypotheses_path: Path, prefix: str = 'H', limit: int = 5) -> str:
    try:
        rows = json.loads(hypotheses_path.read_text(encoding='utf-8'))
    except Exception:
        return 'Не удалось прочитать hypotheses_ranked.json'
    if not isinstance(rows, list) or not rows:
        return 'Гипотезы не были сгенерированы.'
    lines = []
    for row in rows[:limit]:
        hyp = row.get('hypothesis') if isinstance(row.get('hypothesis'), dict) else {}
        title = str(hyp.get('title') or '(без названия)')
        lines.append(f"- **{prefix}-{int(row.get('rank') or 0):03d}** — {title}  ")
    return "\n".join(lines)


def _deep_update(base: dict, extra: dict) -> dict:
    result = dict(base or {})
    for key, value in (extra or {}).items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = _deep_update(result.get(key) or {}, value)
        else:
            result[key] = value
    return result


def _task3_dual_default_setup() -> dict:
    return {
        'input_mode': 'query',
        'task1_yaml_path': '',
        'processed_path': '',
        'commands_file_path': '',
        'query': '',
        'identifiers': [],
        'commands_text': '',
        'expert': {
            'last_name': '',
            'first_name': '',
            'patronymic': '-',
        },
        'domain_id': 'science',
        'out_dir': str(_task3_default_out_dir()),
        'search_limit': 25,
        'top_papers': 12,
        'top_hypotheses': 8,
        'candidate_top_k': 16,
        'top_pairs': 8,
        'annoy_n_trees': 32,
        'annoy_top_k': 6,
        'include_multimodal': True,
        'run_vlm': True,
        'edge_mode': 'auto',
        'link_prediction_backend': 'auto',
        'hf_offline': _task3_is_kaggle_runtime(),
        'create_offline_form': True,
        'create_expert_bundle': True,
        'auto_download_offline': False,
        'auto_download_bundle': False,
        'auto_download_owner_key': False,
        'model_a': {
            'owner_label': 'base_local_model',
            'vlm_backend': 'qwen2_vl',
            'vlm_model_id': TASK3_DEFAULT_LOCAL_VLM_MODEL,
            'local_text_model': '',
        },
        'model_b': {
            'owner_label': 'finetuned_local_model',
            'vlm_backend': 'qwen2_vl',
            'vlm_model_id': '',
            'local_text_model': '',
        },
    }


def _coerce_int(value, default):
    try:
        return int(value)
    except Exception:
        return int(default)


def _normalize_task3_dual_setup(raw: dict | None) -> dict:
    raw = dict(raw or {})
    allowed_input_modes = {'yaml', 'query', 'commands'}
    allowed_vlm_backends = {'qwen2_vl', 'qwen3_vl', 'llava', 'phi3_vision', 'auto', 'none'}
    allowed_edge_modes = {'auto', 'cooccurrence', 'paper_chain'}
    allowed_link_backends = {'auto', 'heuristic', 'pygt_temporal', 'tgn'}

    base = _task3_dual_default_setup()
    merged = _deep_update(base, raw)

    merged['input_mode'] = str(merged.get('input_mode') or 'query').strip().lower()
    if merged['input_mode'] not in allowed_input_modes:
        merged['input_mode'] = 'query'

    merged['task1_yaml_path'] = str(merged.get('task1_yaml_path') or '').strip()
    merged['processed_path'] = str(merged.get('processed_path') or '').strip()
    merged['commands_file_path'] = str(merged.get('commands_file_path') or '').strip()
    merged['query'] = str(merged.get('query') or '').strip()
    merged['commands_text'] = str(merged.get('commands_text') or '').strip()
    merged['identifiers'] = [str(x).strip() for x in (merged.get('identifiers') or []) if str(x).strip()]

    merged['domain_id'] = str(merged.get('domain_id') or 'science').strip() or 'science'
    merged['out_dir'] = str(merged.get('out_dir') or 'runs/task3_dual_local_blind_ab').strip() or 'runs/task3_dual_local_blind_ab'
    merged['search_limit'] = _coerce_int(merged.get('search_limit'), 25)
    merged['top_papers'] = max(0, _coerce_int(merged.get('top_papers'), 12))
    merged['top_hypotheses'] = max(1, _coerce_int(merged.get('top_hypotheses'), 8))
    merged['candidate_top_k'] = max(1, _coerce_int(merged.get('candidate_top_k'), 16))
    merged['top_pairs'] = max(1, _coerce_int(merged.get('top_pairs'), 8))
    merged['annoy_n_trees'] = max(1, _coerce_int(merged.get('annoy_n_trees'), 32))
    merged['annoy_top_k'] = max(1, _coerce_int(merged.get('annoy_top_k'), 6))

    merged['include_multimodal'] = _parse_bool(merged.get('include_multimodal'), default=True)
    merged['run_vlm'] = _parse_bool(merged.get('run_vlm'), default=True)
    merged['hf_offline'] = _parse_bool(merged.get('hf_offline'), default=False)
    merged['create_offline_form'] = _parse_bool(merged.get('create_offline_form'), default=True)
    merged['create_expert_bundle'] = _parse_bool(merged.get('create_expert_bundle'), default=True)
    merged['auto_download_offline'] = _parse_bool(merged.get('auto_download_offline'), default=False)
    merged['auto_download_bundle'] = _parse_bool(merged.get('auto_download_bundle'), default=False)
    merged['auto_download_owner_key'] = _parse_bool(merged.get('auto_download_owner_key'), default=False)

    merged['edge_mode'] = str(merged.get('edge_mode') or 'auto').strip()
    if merged['edge_mode'] not in allowed_edge_modes:
        merged['edge_mode'] = 'auto'
    merged['link_prediction_backend'] = str(merged.get('link_prediction_backend') or 'auto').strip()
    if merged['link_prediction_backend'] not in allowed_link_backends:
        merged['link_prediction_backend'] = 'auto'

    expert = merged.get('expert') if isinstance(merged.get('expert'), dict) else {}
    merged['expert'] = {
        'last_name': str(expert.get('last_name') or '').strip(),
        'first_name': str(expert.get('first_name') or '').strip(),
        'patronymic': str(expert.get('patronymic') or '-').strip() or '-',
    }

    for model_key, default_owner in [('model_a', 'base_local_model'), ('model_b', 'finetuned_local_model')]:
        model = merged.get(model_key) if isinstance(merged.get(model_key), dict) else {}
        backend = str(model.get('vlm_backend') or 'qwen2_vl').strip()
        if backend not in allowed_vlm_backends:
            backend = 'qwen2_vl'
        merged[model_key] = {
            'owner_label': str(model.get('owner_label') or default_owner).strip() or default_owner,
            'vlm_backend': backend,
            'vlm_model_id': str(model.get('vlm_model_id') or '').strip(),
            'local_text_model': str(model.get('local_text_model') or '').strip(),
        }

    if not merged['model_a']['vlm_model_id']:
        merged['model_a']['vlm_model_id'] = TASK3_DEFAULT_LOCAL_VLM_MODEL

    return merged


def _build_task3_dual_setup():
    parse_error = None
    merged = _task3_dual_default_setup()

    raw_setup = globals().get('TASK3_DUAL_SETUP')
    if isinstance(raw_setup, dict):
        merged = _deep_update(merged, raw_setup)

    yaml_text = str(globals().get('TASK3_DUAL_SETUP_YAML') or '').strip()
    if yaml_text:
        try:
            yaml_payload = yaml.safe_load(yaml_text) or {}
            if not isinstance(yaml_payload, dict):
                raise ValueError('TASK3_DUAL_SETUP_YAML должен содержать top-level object')
            merged = _deep_update(merged, yaml_payload)
        except Exception as e:
            parse_error = f'{type(e).__name__}: {e}'

    return _normalize_task3_dual_setup(merged), parse_error


TASK3_DUAL_EFFECTIVE_SETUP, TASK3_DUAL_SETUP_PARSE_ERROR = _build_task3_dual_setup()


def _render_message_box(title: str, lines: list[str] | None = None, tone: str = 'neutral', details_html: str = '') -> str:
    palette = {
        'neutral': ('#eef2ff', '#c7d2fe', '#312e81'),
        'running': ('#eff6ff', '#bfdbfe', '#1d4ed8'),
        'success': ('#ecfdf5', '#86efac', '#166534'),
        'warning': ('#fff7ed', '#fdba74', '#9a3412'),
        'danger': ('#fef2f2', '#fca5a5', '#991b1b'),
    }
    bg, border, color = palette.get(tone, palette['neutral'])
    items = ''.join(f'<li>{_escape_html(line)}</li>' for line in (lines or []))
    body = f'<ul style="margin:8px 0 0 18px;">{items}</ul>' if items else ''
    return (
        f'<div style="padding:12px 14px; border-radius:12px; border:1px solid {border}; '
        f'background:{bg}; color:{color}; margin:6px 0;">'
        f'<div style="font-weight:700;">{_escape_html(title)}</div>'
        f'{body}{details_html}</div>'
    )


def _validate_task3dual_snapshot(snapshot: dict) -> dict:
    errors = []
    warnings = []

    if not snapshot.get('boot_ok', TASK3_DUAL_BOOT_OK):
        boot_error = snapshot.get('boot_error') or TASK3_DUAL_BOOT_ERROR or {}
        msg = boot_error.get('message') if isinstance(boot_error, dict) else str(boot_error)
        errors.append(f'Notebook runtime не готов: {msg or "ошибка импорта"}')

    if TASK3_DUAL_SETUP_PARSE_ERROR:
        warnings.append(f'Не удалось разобрать TASK3_DUAL_SETUP_YAML: {TASK3_DUAL_SETUP_PARSE_ERROR}')

    input_mode = str(snapshot.get('input_mode') or 'query').strip()
    trajectory_path = str(snapshot.get('trajectory_path') or '').strip()
    processed_path = str(snapshot.get('processed_path') or '').strip()
    commands_path = str(snapshot.get('commands_path') or '').strip()
    query_text = str(snapshot.get('query') or '').strip()
    commands_text = str(snapshot.get('commands_text') or '').strip()
    identifiers = [str(x).strip() for x in (snapshot.get('identifiers') or []) if str(x).strip()]

    if trajectory_path:
        p = Path(trajectory_path).expanduser()
        if not p.exists():
            errors.append(f'Не найден Task 1 YAML path: {p}')
    if processed_path:
        p = Path(processed_path).expanduser()
        if not p.exists():
            errors.append(f'Не найден processed path: {p}')
        elif p.is_file() and p.suffix.lower() != '.zip':
            errors.append(f'Processed path должен вести к zip или директории, а не к файлу {p.name}')
    if commands_path:
        p = Path(commands_path).expanduser()
        if not p.exists():
            errors.append(f'Не найден commands path: {p}')

    if input_mode == 'yaml' and not (snapshot.get('trajectory_upload_present') or trajectory_path):
        errors.append('Выбран режим Task 1 YAML, но не указан upload или путь к YAML.')
    if input_mode == 'commands' and not (commands_text or snapshot.get('commands_upload_present') or commands_path):
        errors.append('Выбран режим Commands, но commands text / upload / path пусты.')
    if input_mode == 'query' and not (query_text or identifiers or snapshot.get('processed_upload_present') or processed_path):
        errors.append('Для режима Query укажите query и/или identifiers, либо processed ZIP/dir.')

    model_a = snapshot.get('model_a') or {}
    model_b = snapshot.get('model_b') or {}
    if str(model_a.get('vlm_backend') or '').strip() != 'none' and not str(model_a.get('vlm_model_id') or '').strip():
        errors.append('Не указан model/path для Model α.')
    if str(model_b.get('vlm_backend') or '').strip() != 'none' and not str(model_b.get('vlm_model_id') or '').strip():
        errors.append('Не указан model/path для Model β.')

    expert = snapshot.get('expert') or {}
    if not str(expert.get('last_name') or '').strip():
        warnings.append('Не указана фамилия эксперта — submission_id будет менее читаемым.')
    if not str(expert.get('first_name') or '').strip():
        warnings.append('Не указано имя эксперта — submission_id будет менее читаемым.')

    return {
        'ok': not errors,
        'errors': errors,
        'warnings': warnings,
    }

In [8]:
# @title
# Форма dual-local blind A/B (запустите ячейку и заполните поля)

if '_task3_require_validated_setup' not in globals():
    raise RuntimeError('Сначала запустите ячейку быстрого сетапа и отдельную ячейку проверки.')
_task3_require_validated_setup()

from IPython.display import display

_setup = TASK3_DUAL_EFFECTIVE_SETUP

input_mode = W.Dropdown(
    options=[
        ('Task 1 YAML', 'yaml'),
        ('Query + identifiers', 'query'),
        ('Commands', 'commands'),
    ],
    value=_setup['input_mode'],
    description='Input mode',
)

trajectory_path = W.Text(
    value=_setup['task1_yaml_path'],
    description='YAML path',
    placeholder='/content/task1.yaml или /kaggle/input/.../task1.yaml',
    layout=W.Layout(width='100%'),
)
trajectory_upload = W.FileUpload(accept='.yaml,.yml', multiple=False, description='Task1 YAML')

processed_path = W.Text(
    value=_setup['processed_path'],
    description='Processed path',
    placeholder='/content/processed_papers.zip, /content/processed_papers или /kaggle/input/...',
    layout=W.Layout(width='100%'),
)
processed_upload = W.FileUpload(accept='.zip', multiple=False, description='Processed ZIP')

commands_path = W.Text(
    value=_setup['commands_file_path'],
    description='Commands path',
    placeholder='/content/commands.yaml или /kaggle/input/.../commands.yaml',
    layout=W.Layout(width='100%'),
)
commands_upload = W.FileUpload(accept='.txt,.yaml,.yml,.json', multiple=False, description='Commands file')

query = W.Textarea(
    value=_setup['query'],
    description='Query',
    placeholder='temporal knowledge graph multimodal hypothesis generation',
    layout=W.Layout(width='100%', height='90px'),
)
identifiers_text = W.Textarea(
    value='\n'.join(_setup['identifiers']),
    description='IDs',
    placeholder='DOI / URL / identifier — по одному на строку',
    layout=W.Layout(width='100%', height='110px'),
)
commands_text = W.Textarea(
    value=_setup['commands_text'],
    description='Commands',
    placeholder="query: ...\nidentifier: ...\nmodel_a_vlm_model_id: /models/base\nmodel_b_vlm_model_id: /models/tuned\n...",
    layout=W.Layout(width='100%', height='170px'),
)

expert_last = W.Text(description='Фамилия', placeholder='Иванов', value=_setup['expert']['last_name'], layout=W.Layout(width='100%'))
expert_first = W.Text(description='Имя', placeholder='Иван', value=_setup['expert']['first_name'], layout=W.Layout(width='100%'))
expert_pat = W.Text(description='Отчество', placeholder='Иванович или -', value=_setup['expert']['patronymic'], layout=W.Layout(width='100%'))

domain_id = W.Text(value=_setup['domain_id'], description='Domain', layout=W.Layout(width='100%'))
out_dir = W.Text(value=_setup['out_dir'], description='Out dir', layout=W.Layout(width='100%'))
search_limit = W.IntSlider(value=_setup['search_limit'], min=5, max=100, step=5, description='Search limit', continuous_update=False)
top_papers = W.IntSlider(value=_setup['top_papers'], min=0, max=50, step=1, description='Top papers', continuous_update=False)
top_hypotheses = W.IntSlider(value=_setup['top_hypotheses'], min=1, max=20, step=1, description='Top hyps', continuous_update=False)
candidate_top_k = W.IntSlider(value=_setup['candidate_top_k'], min=1, max=50, step=1, description='Cand top-k', continuous_update=False)
top_pairs = W.IntSlider(value=_setup['top_pairs'], min=1, max=20, step=1, description='Top pairs', continuous_update=False)
include_multimodal = W.Checkbox(value=bool(_setup['include_multimodal']), description='Include multimodal chunks')
run_vlm = W.Checkbox(value=bool(_setup['run_vlm']), description='Run VLM on pages / images / tables')
edge_mode = W.Dropdown(options=['auto', 'cooccurrence', 'paper_chain'], value=_setup['edge_mode'], description='Edge mode')
link_prediction_backend = W.Dropdown(options=['auto', 'heuristic', 'pygt_temporal', 'tgn'], value=_setup['link_prediction_backend'], description='Link pred')
annoy_n_trees = W.IntSlider(value=_setup['annoy_n_trees'], min=4, max=64, step=4, description='Annoy trees', continuous_update=False)
annoy_top_k = W.IntSlider(value=_setup['annoy_top_k'], min=1, max=20, step=1, description='Annoy top-k', continuous_update=False)

model_a_owner_label = W.Text(value=_setup['model_a']['owner_label'], description='Owner A', layout=W.Layout(width='100%'))
model_a_vlm_backend = W.Dropdown(
    options=[
        ('qwen2_vl (HF local)', 'qwen2_vl'),
        ('qwen3_vl (HF local)', 'qwen3_vl'),
        ('llava (HF local)', 'llava'),
        ('phi3_vision (HF local)', 'phi3_vision'),
        ('auto', 'auto'),
        ('none', 'none'),
    ],
    value=_setup['model_a']['vlm_backend'],
    description='VLM α',
)
model_a_vlm_model_id = W.Text(value=_setup['model_a']['vlm_model_id'] or TASK3_DEFAULT_LOCAL_VLM_MODEL, description='Model/path α', placeholder='Qwen/Qwen2.5-VL-3B-Instruct или /models/base-vlm', layout=W.Layout(width='100%'))
model_a_local_text_model = W.Text(value=_setup['model_a']['local_text_model'], description='Text α', placeholder='необязательно: локальная text model', layout=W.Layout(width='100%'))

model_b_owner_label = W.Text(value=_setup['model_b']['owner_label'], description='Owner β', layout=W.Layout(width='100%'))
model_b_vlm_backend = W.Dropdown(
    options=[
        ('qwen2_vl (HF local)', 'qwen2_vl'),
        ('qwen3_vl (HF local)', 'qwen3_vl'),
        ('llava (HF local)', 'llava'),
        ('phi3_vision (HF local)', 'phi3_vision'),
        ('auto', 'auto'),
        ('none', 'none'),
    ],
    value=_setup['model_b']['vlm_backend'],
    description='VLM β',
)
model_b_vlm_model_id = W.Text(value=_setup['model_b']['vlm_model_id'], description='Model/path β', placeholder='/models/finetuned-vlm', layout=W.Layout(width='100%'))
model_b_local_text_model = W.Text(value=_setup['model_b']['local_text_model'], description='Text β', placeholder='необязательно: локальная text model', layout=W.Layout(width='100%'))

hf_offline = W.Checkbox(value=bool(_setup['hf_offline']), description='HF offline / local-files mode')
create_offline_form = W.Checkbox(value=bool(_setup['create_offline_form']), description='Build blind offline HTML')
create_expert_bundle = W.Checkbox(value=bool(_setup['create_expert_bundle']), description='Build expert ZIP (without owner key)')
auto_download_offline = W.Checkbox(value=bool(_setup['auto_download_offline']), description='Auto-download offline HTML (Colab)')
auto_download_bundle = W.Checkbox(value=bool(_setup['auto_download_bundle']), description='Auto-download expert ZIP (Colab)')
auto_download_owner_key = W.Checkbox(value=bool(_setup['auto_download_owner_key']), description='Auto-download owner key (Colab)')

apply_setup_btn = W.Button(description='Применить быстрый сетап', button_style='info')
validate_btn = W.Button(description='Проверить конфиг', button_style='success')

status = W.HTML()
validation_html = W.HTML()
summary_html = W.HTML()

overall_progress_label = W.HTML()
overall_progress_bar = W.IntProgress(value=0, min=0, max=100, description='Overall', bar_style='info', layout=W.Layout(width='100%'))
run_progress_label = W.HTML()
run_progress_bar = W.IntProgress(value=0, min=0, max=100, description='Current', bar_style='info', layout=W.Layout(width='100%'))
detail_progress_label = W.HTML()
detail_progress_bar = W.IntProgress(value=0, min=0, max=100, description='Detail', bar_style='info', layout=W.Layout(width='100%'))
progress_log = W.HTML()

out = W.Output(layout=W.Layout(border='1px solid var(--jp-border-color2, #e2e8f0)', padding='8px'))

last_paths_html = W.HTML('<div class="task3-note"><b>Артефакты ещё не созданы.</b></div>')
download_offline_btn = W.Button(description='Скачать offline HTML', button_style='primary')
download_offline_btn.disabled = True
download_bundle_btn = W.Button(description='Скачать expert ZIP', button_style='warning')
download_bundle_btn.disabled = True
download_owner_key_btn = W.Button(description='Скачать owner key', button_style='info')
download_owner_key_btn.disabled = True
artifact_out = W.Output()


def _progress_note_html(text, tone='neutral'):
    palette = {
        'neutral': ('#f8fafc', '#cbd5e1', '#334155'),
        'running': ('#eff6ff', '#93c5fd', '#1d4ed8'),
        'success': ('#ecfdf5', '#86efac', '#166534'),
        'warning': ('#fff7ed', '#fdba74', '#9a3412'),
    }
    bg, border, color = palette.get(tone, palette['neutral'])
    return (
        f'<div style="padding:8px 10px; border-radius:10px; border:1px solid {border}; '
        f'background:{bg}; color:{color}; margin:4px 0; font-weight:600;">{_escape_html(text or "Ожидание")}</div>'
    )


def _set_task3dual_paths(paths):
    global TASK3_DUAL_LAST_ARTIFACTS
    TASK3_DUAL_LAST_ARTIFACTS = paths
    offline_p = paths.get('offline_form') if isinstance(paths, dict) else None
    bundle_p = paths.get('expert_bundle') if isinstance(paths, dict) else None
    owner_p = paths.get('owner_key') if isinstance(paths, dict) else None
    download_offline_btn.disabled = not (offline_p and Path(offline_p).exists())
    download_bundle_btn.disabled = not (bundle_p and Path(bundle_p).exists())
    download_owner_key_btn.disabled = not (owner_p and Path(owner_p).exists())
    parts = []
    if paths:
        for key in [
            'run_alpha_bundle', 'run_beta_bundle', 'manifest_alpha', 'manifest_beta',
            'hypotheses_alpha_json', 'hypotheses_beta_json', 'offline_form', 'expert_bundle', 'owner_key', 'public_manifest'
        ]:
            if paths.get(key):
                parts.append(f'<div><b>{key}</b>: <code>{_escape_html(str(paths[key]))}</code></div>')
    last_paths_html.value = '<div class="task3-note">' + ''.join(parts) + '</div>' if parts else '<div class="task3-note">Нет артефактов.</div>'


def _set_status_box(title: str, lines: list[str] | None = None, tone: str = 'neutral', details_html: str = ''):
    status.value = _render_message_box(title, lines=lines, tone=tone, details_html=details_html)


def _set_validation_box(report: dict | None = None, *, title: str = 'Проверка конфига'):
    report = report or {}
    pieces = []
    if report.get('errors'):
        pieces.append(_render_message_box(title + ' — есть ошибки', lines=report['errors'], tone='warning'))
    if report.get('warnings'):
        pieces.append(_render_message_box(title + ' — есть предупреждения', lines=report['warnings'], tone='neutral'))
    if not report.get('errors') and not report.get('warnings'):
        pieces.append(_render_message_box(title, lines=['Конфиг выглядит корректно, можно запускать следующую ячейку.'], tone='success'))
    validation_html.value = ''.join(pieces)


def _set_task3dual_progress(overall_percent=0, overall_text='Ожидание запуска', run_percent=0, run_text='Ожидание pipeline', detail_percent=0, detail_text='Ожидание деталей', tone='neutral', details=''):
    bar_style = 'success' if tone == 'success' else ('warning' if tone == 'warning' else 'info')
    overall_progress_bar.value = max(0, min(100, int(overall_percent or 0)))
    overall_progress_bar.bar_style = bar_style
    overall_progress_label.value = _progress_note_html(overall_text, tone=tone)

    run_progress_bar.value = max(0, min(100, int(run_percent or 0)))
    run_progress_bar.bar_style = bar_style
    run_progress_label.value = _progress_note_html(run_text, tone=tone)

    detail_progress_bar.value = max(0, min(100, int(detail_percent or 0)))
    detail_progress_bar.bar_style = bar_style
    detail_progress_label.value = _progress_note_html(detail_text, tone=tone)

    progress_log.value = f'<div style="font-size:12px; color:#475569; margin-top:4px;">{_escape_html(details)}</div>' if details else ''


def _collect_form_snapshot() -> dict:
    return {
        'boot_ok': TASK3_DUAL_BOOT_OK,
        'boot_error': TASK3_DUAL_BOOT_ERROR,
        'input_mode': input_mode.value,
        'trajectory_path': trajectory_path.value,
        'trajectory_upload_present': bool(trajectory_upload.value),
        'processed_path': processed_path.value,
        'processed_upload_present': bool(processed_upload.value),
        'commands_path': commands_path.value,
        'commands_upload_present': bool(commands_upload.value),
        'query': query.value,
        'identifiers': _parse_identifier_blob(identifiers_text.value),
        'commands_text': commands_text.value,
        'expert': {
            'last_name': expert_last.value,
            'first_name': expert_first.value,
            'patronymic': expert_pat.value,
        },
        'model_a': {
            'owner_label': model_a_owner_label.value,
            'vlm_backend': model_a_vlm_backend.value,
            'vlm_model_id': model_a_vlm_model_id.value,
            'local_text_model': model_a_local_text_model.value,
        },
        'model_b': {
            'owner_label': model_b_owner_label.value,
            'vlm_backend': model_b_vlm_backend.value,
            'vlm_model_id': model_b_vlm_model_id.value,
            'local_text_model': model_b_local_text_model.value,
        },
    }


def _refresh_setup_from_globals():
    global TASK3_DUAL_EFFECTIVE_SETUP, TASK3_DUAL_SETUP_PARSE_ERROR
    TASK3_DUAL_EFFECTIVE_SETUP, TASK3_DUAL_SETUP_PARSE_ERROR = _build_task3_dual_setup()
    return TASK3_DUAL_EFFECTIVE_SETUP


def _apply_setup_to_widgets(setup=None, *, announce=True):
    setup = setup or _refresh_setup_from_globals()
    input_mode.value = setup['input_mode']
    trajectory_path.value = setup['task1_yaml_path']
    processed_path.value = setup['processed_path']
    commands_path.value = setup['commands_file_path']
    query.value = setup['query']
    identifiers_text.value = '\n'.join(setup['identifiers'])
    commands_text.value = setup['commands_text']

    expert_last.value = setup['expert']['last_name']
    expert_first.value = setup['expert']['first_name']
    expert_pat.value = setup['expert']['patronymic']

    domain_id.value = setup['domain_id']
    out_dir.value = setup['out_dir']
    search_limit.value = int(setup['search_limit'])
    top_papers.value = int(setup['top_papers'])
    top_hypotheses.value = int(setup['top_hypotheses'])
    candidate_top_k.value = int(setup['candidate_top_k'])
    top_pairs.value = int(setup['top_pairs'])
    annoy_n_trees.value = int(setup['annoy_n_trees'])
    annoy_top_k.value = int(setup['annoy_top_k'])
    include_multimodal.value = bool(setup['include_multimodal'])
    run_vlm.value = bool(setup['run_vlm'])
    edge_mode.value = setup['edge_mode']
    link_prediction_backend.value = setup['link_prediction_backend']

    model_a_owner_label.value = setup['model_a']['owner_label']
    model_a_vlm_backend.value = setup['model_a']['vlm_backend']
    model_a_vlm_model_id.value = setup['model_a']['vlm_model_id'] or TASK3_DEFAULT_LOCAL_VLM_MODEL
    model_a_local_text_model.value = setup['model_a']['local_text_model']

    model_b_owner_label.value = setup['model_b']['owner_label']
    model_b_vlm_backend.value = setup['model_b']['vlm_backend']
    model_b_vlm_model_id.value = setup['model_b']['vlm_model_id']
    model_b_local_text_model.value = setup['model_b']['local_text_model']

    hf_offline.value = bool(setup['hf_offline'])
    create_offline_form.value = bool(setup['create_offline_form'])
    create_expert_bundle.value = bool(setup['create_expert_bundle'])
    auto_download_offline.value = bool(setup['auto_download_offline'])
    auto_download_bundle.value = bool(setup['auto_download_bundle'])
    auto_download_owner_key.value = bool(setup['auto_download_owner_key'])

    report = _validate_task3dual_snapshot(_collect_form_snapshot())
    _set_validation_box(report, title='Проверка после применения быстрого сетапа')
    if announce:
        note_lines = ['Значения из верхней ячейки применены к форме.']
        if TASK3_DUAL_SETUP_PARSE_ERROR:
            note_lines.append('YAML override не разобрался — использованы значения из словаря и дефолты.')
        _set_status_box('Форма синхронизирована', lines=note_lines, tone='neutral')


def _download_offline(_):
    artifact_out.clear_output()
    paths = TASK3_DUAL_LAST_ARTIFACTS or {}
    target = Path(paths.get('offline_form')) if paths.get('offline_form') else None
    if target is None or not target.exists():
        _set_status_box('Offline HTML пока не готов', tone='warning')
        return
    _download_file_if_possible(target, status_widget=status)


def _download_bundle(_):
    artifact_out.clear_output()
    paths = TASK3_DUAL_LAST_ARTIFACTS or {}
    target = Path(paths.get('expert_bundle')) if paths.get('expert_bundle') else None
    if target is None or not target.exists():
        _set_status_box('Expert ZIP пока не готов', tone='warning')
        return
    _download_file_if_possible(target, status_widget=status)


def _download_owner_key(_):
    artifact_out.clear_output()
    paths = TASK3_DUAL_LAST_ARTIFACTS or {}
    target = Path(paths.get('owner_key')) if paths.get('owner_key') else None
    if target is None or not target.exists():
        _set_status_box('Owner key пока не готов', tone='warning')
        return
    _download_file_if_possible(target, status_widget=status)


def _on_apply_setup(_):
    _apply_setup_to_widgets(_refresh_setup_from_globals(), announce=True)


def _on_validate(_):
    report = _validate_task3dual_snapshot(_collect_form_snapshot())
    _set_validation_box(report, title='Проверка текущего состояния формы')
    if report['ok']:
        _set_status_box('Конфиг прошёл проверку', lines=['Можно запускать следующую ячейку.'], tone='success')
    else:
        _set_status_box('Исправьте конфиг перед запуском', lines=report['errors'], tone='warning')
    if report.get('warnings') and report['ok']:
        _set_status_box('Конфиг прошёл проверку с предупреждениями', lines=report['warnings'], tone='neutral')


download_offline_btn.on_click(_download_offline)
download_bundle_btn.on_click(_download_bundle)
download_owner_key_btn.on_click(_download_owner_key)
apply_setup_btn.on_click(_on_apply_setup)
validate_btn.on_click(_on_validate)

left = W.VBox([
    W.HTML('<h3>Входные данные</h3>'),
    W.HTML('<div style="font-size:12px; color:#475569;">Можно использовать и пути, и upload. Upload имеет приоритет над путём.</div>'),
    input_mode,
    trajectory_path,
    trajectory_upload,
    processed_path,
    processed_upload,
    query,
    identifiers_text,
    commands_text,
    commands_path,
    commands_upload,
])
right = W.VBox([
    W.HTML('<h3>Эксперт и общие параметры</h3>'),
    expert_last,
    expert_first,
    expert_pat,
    domain_id,
    out_dir,
    search_limit,
    top_papers,
    top_hypotheses,
    candidate_top_k,
    top_pairs,
    include_multimodal,
    run_vlm,
    edge_mode,
    link_prediction_backend,
    annoy_n_trees,
    annoy_top_k,
])
model_alpha_box = W.VBox([
    W.HTML('<h3>Model α (обычно baseline / из коробки)</h3>'),
    model_a_owner_label,
    model_a_vlm_backend,
    model_a_vlm_model_id,
    model_a_local_text_model,
])
model_beta_box = W.VBox([
    W.HTML('<h3>Model β (обычно finetuned)</h3>'),
    model_b_owner_label,
    model_b_vlm_backend,
    model_b_vlm_model_id,
    model_b_local_text_model,
])
common_box = W.VBox([
    W.HTML('<h3>Blind review и сохранение</h3>'),
    hf_offline,
    create_offline_form,
    create_expert_bundle,
    auto_download_offline,
    auto_download_bundle,
    auto_download_owner_key,
])

progress_box = W.VBox([
    W.HTML('<h3>Прогресс выполнения</h3>'),
    overall_progress_label,
    overall_progress_bar,
    run_progress_label,
    run_progress_bar,
    detail_progress_label,
    detail_progress_bar,
    progress_log,
])

ui = W.VBox([
    W.HBox([apply_setup_btn, validate_btn, download_offline_btn, download_bundle_btn, download_owner_key_btn]),
    W.HBox([left, right], layout=W.Layout(align_items='flex-start')),
    W.HBox([model_alpha_box, model_beta_box, common_box], layout=W.Layout(align_items='flex-start')),
    progress_box,
    status,
    validation_html,
    summary_html,
    last_paths_html,
    artifact_out,
    out,
])

# Headless smoke defaults for automated validation
if os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE') == '1':
    _smoke_query = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_QUERY') or 'offline dual local model blind review').strip()
    _smoke_processed = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_PROCESSED_DIR') or '').strip()
    _smoke_out_dir = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_OUT_DIR') or 'runs/task3_dual_local_blind_ab_smoke').strip()
    _smoke_model_a = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_MODEL_A') or 'mock-local-model-a').strip() or 'mock-local-model-a'
    _smoke_model_b = (os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_MODEL_B') or 'mock-local-model-b').strip() or 'mock-local-model-b'

    input_mode.value = 'commands'
    query.value = _smoke_query
    commands_text.value = (
        f"query: {_smoke_query}\n"
        + (f"processed_dir: {_smoke_processed}\n" if _smoke_processed else "")
        + f"out_dir: {_smoke_out_dir}\n"
        + "top_papers: 0\n"
        + "top_hypotheses: 3\n"
        + "candidate_top_k: 5\n"
        + "top_pairs: 3\n"
        + "include_multimodal: true\n"
        + "run_vlm: false\n"
        + "edge_mode: cooccurrence\n"
        + "link_prediction_backend: heuristic\n"
        + "hf_offline: true\n"
        + "model_a_vlm_backend: none\n"
        + f"model_a_vlm_model_id: {_smoke_model_a}\n"
        + "model_b_vlm_backend: none\n"
        + f"model_b_vlm_model_id: {_smoke_model_b}\n"
    )
    processed_path.value = _smoke_processed
    expert_last.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_EXPERT_LAST', 'Smoke')
    expert_first.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_EXPERT_FIRST', 'Validation')
    expert_pat.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_EXPERT_PAT', '-')
    domain_id.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_DOMAIN_ID', 'science')
    out_dir.value = _smoke_out_dir
    search_limit.value = 5
    top_papers.value = 0
    top_hypotheses.value = 3
    candidate_top_k.value = 5
    top_pairs.value = 3
    include_multimodal.value = True
    run_vlm.value = False
    edge_mode.value = 'cooccurrence'
    link_prediction_backend.value = 'heuristic'
    model_a_owner_label.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_OWNER_A', 'base_local_model')
    model_a_vlm_backend.value = 'none'
    model_a_vlm_model_id.value = _smoke_model_a
    model_a_local_text_model.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_TEXT_A', '')
    model_b_owner_label.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_OWNER_B', 'finetuned_local_model')
    model_b_vlm_backend.value = 'none'
    model_b_vlm_model_id.value = _smoke_model_b
    model_b_local_text_model.value = os.environ.get('TASK3_DUAL_NOTEBOOK_SMOKE_TEXT_B', '')
    hf_offline.value = True
    create_offline_form.value = True
    create_expert_bundle.value = True
    auto_download_offline.value = False
    auto_download_bundle.value = False
    auto_download_owner_key.value = False

_set_task3dual_paths({})
_set_task3dual_progress(0, 'Ожидание запуска', 0, 'Pipeline ещё не стартовал', 0, 'Нет активного шага', tone='neutral')
_apply_setup_to_widgets(_setup, announce=False)

_initial_report = _validate_task3dual_snapshot(_collect_form_snapshot())
_set_validation_box(_initial_report, title='Проверка стартового состояния формы')
if TASK3_DUAL_BOOT_OK:
    _set_status_box('Форма готова', lines=['Можно проверить конфиг или сразу запускать следующую ячейку.'], tone='neutral')
else:
    boot_message = TASK3_DUAL_BOOT_ERROR.get('message') if isinstance(TASK3_DUAL_BOOT_ERROR, dict) else str(TASK3_DUAL_BOOT_ERROR)
    _set_status_box('Runtime загружен не полностью', lines=[boot_message or 'Ошибка импорта'], tone='warning')

display(ui)

In [ ]:
# @title
# Запуск dual-local blind A/B из отдельной ячейки
# Важно: после заполнения формы выше запускайте именно ЭТУ ячейку.



if '_task3_require_validated_setup' not in globals():
    raise RuntimeError('Сначала запустите ячейку быстрого сетапа и отдельную ячейку проверки.')
_task3_require_validated_setup()

def _make_variant_progress_callback(variant_label: str, overall_start: int, overall_end: int):
    span = max(1, overall_end - overall_start)

    def _callback(payload: dict):
        stage_current = int(payload.get('current') or 0)
        stage_total = max(1, int(payload.get('total') or 1))
        run_percent = int(round((stage_current / stage_total) * 100))

        item_current = payload.get('item_current')
        item_total = payload.get('item_total')
        page_current = payload.get('page_current')
        page_total = payload.get('page_total')

        detail_current = page_current if page_current is not None else item_current
        detail_total = page_total if page_total is not None else item_total
        detail_percent = 0
        if detail_current is not None and detail_total:
            try:
                detail_percent = int(round((int(detail_current) / max(1, int(detail_total))) * 100))
            except Exception:
                detail_percent = 0

        details_parts = [f'variant {variant_label}', f"stage={payload.get('stage') or 'unknown'}"]
        if item_current is not None and item_total:
            details_parts.append(f'items {item_current}/{item_total}')
        if page_current is not None and page_total:
            details_parts.append(f'pages {page_current}/{page_total}')
        if payload.get('paper_title'):
            details_parts.append(str(payload.get('paper_title'))[:120])

        overall_percent = int(round(overall_start + (run_percent / 100.0) * span))
        _set_task3dual_progress(
            overall_percent=overall_percent,
            overall_text=f'Выполняется run {variant_label}',
            run_percent=run_percent,
            run_text=str(payload.get('message') or f'Run {variant_label}'),
            detail_percent=detail_percent,
            detail_text='Текущий paper/page/candidate progress',
            tone='running',
            details=' · '.join(details_parts),
        )

    return _callback


def run_task3_dual_local_ab_from_form():
    global TASK3_DUAL_LAST_RESULT, TASK3_DUAL_LAST_TASK_META, TASK3_DUAL_LAST_ERROR
    with out:
        clear_output()
        TASK3_DUAL_LAST_RESULT = None
        TASK3_DUAL_LAST_ERROR = None
        summary_html.value = ''

        snapshot = _collect_form_snapshot()
        report = _validate_task3dual_snapshot(snapshot)
        _set_validation_box(report, title='Проверка перед запуском')
        if not report['ok']:
            _set_status_box('Исправьте конфиг перед запуском', lines=report['errors'], tone='warning')
            _set_task3dual_progress(0, 'Запуск не начался', 0, 'Есть ошибки в конфиге', 0, 'Исправьте поля и перезапустите ячейку', tone='warning')
            return None

        if not TASK3_DUAL_BOOT_OK or prepare_task3_hypothesis_bundle is None:
            boot_message = TASK3_DUAL_BOOT_ERROR.get('message') if isinstance(TASK3_DUAL_BOOT_ERROR, dict) else 'runtime не инициализирован'
            _set_status_box('Запуск недоступен', lines=[boot_message], tone='warning')
            _set_task3dual_progress(0, 'Запуск недоступен', 0, 'runtime не готов', 0, 'Исправьте ошибку импорта', tone='warning')
            return None

        try:
            _set_status_box('Task 3 dual-local blind A/B: подготовка запуска', tone='running')
            _set_task3dual_progress(2, 'Подготовка запуска', 0, 'Собираю входные файлы', 0, 'Создаю рабочую директорию и читаю конфиг', tone='running')

            workdir = Path(tempfile.mkdtemp(prefix='task3_dual_local_ab_'))

            trajectory_from_upload = _materialize_upload(trajectory_upload.value, workdir, 'task1.yaml')
            trajectory_from_path = _materialize_local_file(trajectory_path.value, workdir, 'task1.yaml') if trajectory_from_upload is None else None
            trajectory_path_effective = trajectory_from_upload or trajectory_from_path

            commands_file_uploaded = _materialize_upload(commands_upload.value, workdir, 'commands.txt')
            commands_file_from_path = _materialize_local_file(commands_path.value, workdir, 'commands.txt') if commands_file_uploaded is None else None
            commands_file_path = commands_file_uploaded or commands_file_from_path

            processed_dir = _extract_processed_source(processed_upload.value, processed_path.value, workdir)

            commands = _merged_commands_from_sources(commands_text.value, commands_upload.value, commands_path.value)
            trajectory_doc = _load_yaml_if_present(trajectory_path_effective) if trajectory_path_effective else {}
            trajectory_for_run = trajectory_path_effective if input_mode.value == 'yaml' else None
            task_meta = _task3_task_meta_from_widgets(trajectory_doc if trajectory_for_run else {})
            TASK3_DUAL_LAST_TASK_META = task_meta

            inline_identifiers = _parse_identifier_blob(identifiers_text.value)
            identifiers = list(inline_identifiers)
            identifiers.extend(commands.get('identifiers') or [])
            deduped = []
            seen = set()
            for item in identifiers:
                item = str(item).strip()
                if not item or item in seen:
                    continue
                seen.add(item)
                deduped.append(item)
            identifiers = deduped

            effective_query = str(query.value or '').strip()
            if not effective_query:
                effective_query = str(commands.get('query') or '').strip()
            if not effective_query and trajectory_doc:
                effective_query = str(trajectory_doc.get('topic') or '').strip()
            if not effective_query and processed_dir is not None:
                effective_query = 'offline dual local model blind review'

            if input_mode.value == 'yaml' and trajectory_path_effective is None:
                raise ValueError('Выбран режим Task 1 YAML, но YAML-файл не найден ни в upload, ни по path.')
            if input_mode.value == 'commands' and not (commands_text.value.strip() or commands_file_path):
                raise ValueError('Выбран режим Commands, но commands text/file/path пустой.')
            if input_mode.value == 'query' and not effective_query and not identifiers and processed_dir is None:
                raise ValueError('Введите query и/или identifiers, либо укажите processed ZIP/dir.')

            if processed_dir is None and commands.get('processed_dir'):
                candidate = Path(str(commands.get('processed_dir'))).expanduser()
                if candidate.exists():
                    processed_dir = candidate

            selected_domain_id = str(commands.get('domain_id') or domain_id.value or 'science').strip() or 'science'
            selected_out_dir = _task3_resolve_output_dir(str(commands.get('out_dir') or out_dir.value or str(_task3_default_out_dir())))
            selected_top_pairs = int(commands.get('top_pairs') or top_pairs.value)
            include_mm_value = include_multimodal.value if commands.get('include_multimodal') is None else bool(commands['include_multimodal'])
            run_vlm_value = run_vlm.value if commands.get('run_vlm') is None else bool(commands['run_vlm'])
            hf_offline_value = hf_offline.value if commands.get('hf_offline') is None else bool(commands['hf_offline'])

            if hf_offline_value:
                os.environ['HF_HUB_OFFLINE'] = '1'
                os.environ['TRANSFORMERS_OFFLINE'] = '1'
                os.environ.setdefault('HF_DATASETS_OFFLINE', '1')
            else:
                os.environ.pop('HF_HUB_OFFLINE', None)
                os.environ.pop('TRANSFORMERS_OFFLINE', None)

            model_a_cfg = {
                'owner_label': str(commands.get('model_a_owner_label') or model_a_owner_label.value or 'base_local_model').strip() or 'base_local_model',
                'vlm_backend': str(commands.get('model_a_vlm_backend') or model_a_vlm_backend.value or 'qwen2_vl').strip() or 'qwen2_vl',
                'vlm_model_id': str(commands.get('model_a_vlm_model_id') or model_a_vlm_model_id.value or '').strip(),
                'local_text_model': str(commands.get('model_a_local_text_model') or model_a_local_text_model.value or '').strip() or None,
            }
            model_b_cfg = {
                'owner_label': str(commands.get('model_b_owner_label') or model_b_owner_label.value or 'finetuned_local_model').strip() or 'finetuned_local_model',
                'vlm_backend': str(commands.get('model_b_vlm_backend') or model_b_vlm_backend.value or 'qwen2_vl').strip() or 'qwen2_vl',
                'vlm_model_id': str(commands.get('model_b_vlm_model_id') or model_b_vlm_model_id.value or '').strip(),
                'local_text_model': str(commands.get('model_b_local_text_model') or model_b_local_text_model.value or '').strip() or None,
            }

            if not model_a_cfg['vlm_model_id'] and model_a_cfg['vlm_backend'] != 'none':
                raise ValueError('Не указан model/path для Model α (model_a_vlm_model_id).')
            if not model_b_cfg['vlm_model_id'] and model_b_cfg['vlm_backend'] != 'none':
                raise ValueError('Не указан model/path для Model β (model_b_vlm_model_id).')
            if model_a_cfg['vlm_backend'] == 'none':
                run_vlm_value = False
            if model_b_cfg['vlm_backend'] == 'none':
                run_vlm_value = False

            display(Markdown(f"""
# Запуск dual-local blind A/B
- **input_mode**: `{input_mode.value}`
- **effective_query**: `{effective_query or '—'}`
- **trajectory**: `{str(trajectory_for_run) if trajectory_for_run else '—'}`
- **commands_file**: `{str(commands_file_path) if commands_file_path else '—'}`
- **identifiers**: `{len(identifiers)}`
- **processed_dir (initial)**: `{str(processed_dir) if processed_dir else '—'}`
- **domain_id**: `{selected_domain_id}`
- **HF offline**: `{hf_offline_value}`
- **progress callback support**: `{TASK3_DUAL_PIPELINE_SUPPORTS_PROGRESS}`
- **Model α backend**: `{model_a_cfg['vlm_backend']}`
- **Model α path/id**: `{model_a_cfg['vlm_model_id'] or '—'}`
- **Model α local text**: `{model_a_cfg['local_text_model'] or '—'}`
- **Model β backend**: `{model_b_cfg['vlm_backend']}`
- **Model β path/id**: `{model_b_cfg['vlm_model_id'] or '—'}`
- **Model β local text**: `{model_b_cfg['local_text_model'] or '—'}`
"""))

            common_kwargs = dict(
                trajectory=trajectory_for_run,
                query=effective_query,
                identifiers=identifiers,
                domain_id=selected_domain_id,
                search_limit=int(commands.get('search_limit') or search_limit.value),
                top_papers=int(commands.get('top_papers') or top_papers.value),
                top_hypotheses=int(commands.get('top_hypotheses') or top_hypotheses.value),
                candidate_top_k=int(commands.get('candidate_top_k') or candidate_top_k.value),
                include_multimodal=bool(include_mm_value),
                run_vlm=bool(run_vlm_value),
                edge_mode=str(commands.get('edge_mode') or edge_mode.value),
                link_prediction_backend=str(commands.get('link_prediction_backend') or link_prediction_backend.value),
                annoy_n_trees=int(annoy_n_trees.value),
                annoy_top_k=int(annoy_top_k.value),
                llm_provider=None,
                llm_model=None,
                g4f_model=None,
            )

            _set_status_box('Шаг 1/3. Запуск Model α', tone='running')
            _set_task3dual_progress(5, 'Старт run α', 0, 'Подготавливаю pipeline', 0, 'Ожидаю первый progress callback', tone='running')
            alpha_kwargs = dict(
                **common_kwargs,
                processed_dir=processed_dir,
                out_dir=selected_out_dir / 'variant_alpha',
                local_model=model_a_cfg['local_text_model'],
                vlm_backend=model_a_cfg['vlm_backend'],
                vlm_model_id=model_a_cfg['vlm_model_id'],
            )
            if TASK3_DUAL_PIPELINE_SUPPORTS_PROGRESS:
                alpha_kwargs['progress_callback'] = _make_variant_progress_callback('α', 5, 48)
            bundle_alpha = prepare_task3_hypothesis_bundle(**alpha_kwargs)

            shared_processed_dir = processed_dir if processed_dir is not None else (Path(bundle_alpha.bundle_dir) / 'processed_papers')
            if shared_processed_dir is None or not Path(shared_processed_dir).exists():
                raise FileNotFoundError('После run α не удалось определить shared processed_papers directory.')

            _set_status_box('Шаг 2/3. Запуск Model β на тех же processed papers', tone='running')
            _set_task3dual_progress(50, 'Старт run β', 0, 'Подготавливаю pipeline', 0, 'Используются те же processed papers', tone='running')
            beta_kwargs = dict(
                **common_kwargs,
                processed_dir=Path(shared_processed_dir),
                out_dir=selected_out_dir / 'variant_beta',
                local_model=model_b_cfg['local_text_model'],
                vlm_backend=model_b_cfg['vlm_backend'],
                vlm_model_id=model_b_cfg['vlm_model_id'],
            )
            if TASK3_DUAL_PIPELINE_SUPPORTS_PROGRESS:
                beta_kwargs['progress_callback'] = _make_variant_progress_callback('β', 50, 93)
            bundle_beta = prepare_task3_hypothesis_bundle(**beta_kwargs)

            manifest_alpha = json.loads(Path(bundle_alpha.manifest_path).read_text(encoding='utf-8'))
            manifest_beta = json.loads(Path(bundle_beta.manifest_path).read_text(encoding='utf-8'))

            review_assets = None
            expert_bundle_path = None
            if bool(create_offline_form.value) or bool(create_expert_bundle.value):
                _set_status_box('Шаг 3/3. Сборка blind offline review', tone='running')
                _set_task3dual_progress(94, 'Сборка blind review', 100, 'Готовлю offline HTML и служебные файлы', 0, 'Финальная упаковка артефактов', tone='running')
                review_assets = build_task3_dual_model_offline_review_package(
                    manifest_alpha,
                    manifest_beta,
                    task_meta,
                    top_pairs=selected_top_pairs,
                    model_a_descriptor=model_a_cfg,
                    model_b_descriptor=model_b_cfg,
                )
            if bool(create_expert_bundle.value):
                expert_bundle_path = build_task3_dual_model_expert_bundle(
                    manifest_alpha,
                    manifest_beta,
                    task_meta,
                    top_pairs=selected_top_pairs,
                    model_a_descriptor=model_a_cfg,
                    model_b_descriptor=model_b_cfg,
                )

            paths = {
                'run_alpha_bundle': bundle_alpha.bundle_dir,
                'run_beta_bundle': bundle_beta.bundle_dir,
                'manifest_alpha': bundle_alpha.manifest_path,
                'manifest_beta': bundle_beta.manifest_path,
                'hypotheses_alpha_json': bundle_alpha.hypotheses_path,
                'hypotheses_beta_json': bundle_beta.hypotheses_path,
                'hypotheses_alpha_md': Path(bundle_alpha.bundle_dir) / 'hypotheses_ranked.md',
                'hypotheses_beta_md': Path(bundle_beta.bundle_dir) / 'hypotheses_ranked.md',
                'offline_form': review_assets.offline_html_path if review_assets else None,
                'owner_key': review_assets.owner_mapping_path if review_assets else None,
                'public_manifest': review_assets.public_manifest_path if review_assets else None,
                'expert_bundle': expert_bundle_path,
                'run_manifest': run_manifest_path,
                'kaggle_export_zip': kaggle_export_zip,
            }
            _set_task3dual_paths(paths)

            top_alpha = _top_hypotheses_summary(Path(bundle_alpha.hypotheses_path), prefix='α')
            top_beta = _top_hypotheses_summary(Path(bundle_beta.hypotheses_path), prefix='β')
            _set_status_box(
                'Dual-local blind A/B завершён',
                lines=[
                    f'Run α: {bundle_alpha.bundle_dir}',
                    f'Run β: {bundle_beta.bundle_dir}',
                    f'Offline HTML: {str(review_assets.offline_html_path) if review_assets else "—"}',
                    f'Owner key: {str(review_assets.owner_mapping_path) if review_assets else "—"}',
                    f'Expert ZIP: {str(expert_bundle_path) if expert_bundle_path else "—"}',
                    f'Run manifest: {str(run_manifest_path)}',
                    f'Kaggle export ZIP: {str(kaggle_export_zip) if kaggle_export_zip else "—"}',
                ],
                tone='success',
            )
            _set_task3dual_progress(100, 'Dual-local blind A/B завершён', 100, 'Оба прогона и blind review готовы', 100, 'Артефакты сохранены', tone='success')

            summary_html.value = (
                '<div>'
                f'<b>Topic:</b> {_escape_html(task_meta.get("topic") or manifest_alpha.get("query") or "—")}<br>'
                f'<b>Submission:</b> <code>{_escape_html(task_meta.get("submission_id") or "—")}</code><br>'
                f'<b>Shared processed dir:</b> <code>{_escape_html(str(shared_processed_dir))}</code><br>'
                f'<b>Blind offline HTML:</b> <code>{_escape_html(str(review_assets.offline_html_path) if review_assets else "—")}</code><br>'
                f'<b>Owner key:</b> <code>{_escape_html(str(review_assets.owner_mapping_path) if review_assets else "—")}</code><br>'
                f'<b>Expert ZIP:</b> <code>{_escape_html(str(expert_bundle_path) if expert_bundle_path else "—")}</code><br>'
                f'<b>Run manifest:</b> <code>{_escape_html(str(run_manifest_path))}</code><br>'
                f'<b>Kaggle export ZIP:</b> <code>{_escape_html(str(kaggle_export_zip) if kaggle_export_zip else "—")}</code><br>'
                '<hr><b>Top hypotheses α</b><br>' + top_alpha.replace(chr(10), '<br>') +
                '<hr><b>Top hypotheses β</b><br>' + top_beta.replace(chr(10), '<br>') +
                '</div>'
            )

            display(Markdown('## Top hypotheses — Model α'))
            display(Markdown(top_alpha))
            display(Markdown('## Top hypotheses — Model β'))
            display(Markdown(top_beta))

            if auto_download_offline.value and review_assets:
                _download_file_if_possible(Path(review_assets.offline_html_path), status_widget=status)
            if auto_download_bundle.value and expert_bundle_path:
                _download_file_if_possible(Path(expert_bundle_path), status_widget=status)
            if auto_download_owner_key.value and review_assets:
                _download_file_if_possible(Path(review_assets.owner_mapping_path), status_widget=status)

            result = {
                'bundle_alpha': bundle_alpha,
                'bundle_beta': bundle_beta,
                'review_assets': review_assets,
                'expert_bundle': expert_bundle_path,
                'run_manifest': run_manifest_path,
                'kaggle_export_zip': kaggle_export_zip,
            }
            TASK3_DUAL_LAST_RESULT = result
            return result
        except Exception as e:
            trace = traceback.format_exc(limit=8)
            TASK3_DUAL_LAST_ERROR = {
                'type': type(e).__name__,
                'message': str(e),
                'traceback': trace,
            }
            details_html = (
                '<details style="margin-top:8px;">'
                '<summary>Технические детали</summary>'
                f'<pre style="white-space:pre-wrap; font-size:12px;">{_escape_html(trace)}</pre>'
                '</details>'
            )
            _set_status_box(
                'Запуск остановлен с ошибкой',
                lines=[
                    f'{type(e).__name__}: {e}',
                    'Проверьте пути к YAML / processed / commands.',
                    'Проверьте model/path α и β.',
                    'При необходимости поправьте верхнюю ячейку быстрого сетапа и снова запустите форму.',
                ],
                tone='warning',
                details_html=details_html,
            )
            _set_task3dual_progress(
                overall_percent=overall_progress_bar.value,
                overall_text='Запуск остановлен',
                run_percent=run_progress_bar.value,
                run_text='Смотрите сообщение об ошибке',
                detail_percent=detail_progress_bar.value,
                detail_text='Исправьте конфиг и перезапустите ячейку',
                tone='warning',
                details=f'{type(e).__name__}: {e}',
            )
            summary_html.value = _render_message_box(
                'Что проверить в первую очередь',
                lines=[
                    'Task 1 YAML path / upload',
                    'Processed ZIP path или директория',
                    'Commands path / текст commands',
                    'Model/path α и β',
                    'HF offline mode при локальных моделях',
                ],
                tone='warning',
            )
            return None
        finally:
            gc.collect()


TASK3_DUAL_LAST_RESULT = run_task3_dual_local_ab_from_form()

# Что сохраняется после запуска

В типичном случае у вас появятся:

- два отдельных Task 3 run directory (`variant_alpha/...` и `variant_beta/...`),
- blind offline HTML для эксперта,
- owner-only key file,
- expert ZIP без раскрытия owner key,
- ranked hypotheses JSON/MD для обеих модельных конфигураций.

## Практический совет
Если статьи уже были заранее скачаны и распарсены, используйте `processed_papers.zip`. Тогда второй прогон будет быстрее, а сравнение — стабильнее.


### Что меняется на Kaggle

Если `out_dir` находится внутри `\/kaggle\/working`, то после `Save Version` там будут лежать обе директории `variant_alpha` / `variant_beta`, blind offline HTML, owner key, expert ZIP, а также `task3_dual_run_manifest.json`. Дополнительно notebook делает zip-архив с артефактами рядом с каталогом запуска.


# FAQ

## Что указывать в `model/path α` и `model/path β`?
Можно указывать:
- Hugging Face model id,
- локальный путь к директории модели,
- заранее скачанную/закешированную модель.

## Когда включать `HF offline / local-files mode`?
Когда модели уже лежат локально или в кеше и вы хотите запретить сетевые обращения к Hugging Face Hub.

## Почему owner key хранится отдельно?
Потому что blind review должен скрывать от эксперта, какая анонимная система соответствует baseline и какая — finetuned модели.

## Что будет, если обе модели используют одну и ту же локальную text model?
Это допустимо. Тогда основное различие будет приходиться на VLM backend/model path и его влияние на multimodal evidence extraction и последующую формулировку гипотез.
